---

MRI analysis 

---

In [1]:
# autoload
%load_ext autoreload
%autoreload 2

# load relevant pgl classes
from pgl import pgl, pglDatabase, pglActions, pglFlywheel, pglAFNI, pglActionsMRI, pglActionsMNE, pglActionsFixes

---

Load behavioral data

---

In [ ]:
pgl.importSettings()

In [ ]:
newSession = pglActions.loadSession.execute()
if newSession: 
    session = newSession
    session.print()

---

Fix missing volume triggers

---

In [ ]:
fixVolumeTimes = pglActionsFixes.fixVolumeTimes()
newSession = fixVolumeTimes.execute(session=session)
if newSession: 
    session = newSession
    session.print()

In [ ]:
session.display()

In [ ]:
session.print()

---

Load data from flywheel

---

In [ ]:
newSession = pglActions.loadFlywheel().execute(session=session, showScanCounts=False)
if newSession: 
    session = newSession
    session.print()

In [ ]:
session.mri.print(match='nsdmini')

---

Align MRI scans with behavior/experiment session data

---

In [ ]:
session.setScanRunMatches([(4, 0), (5, 1), (6, 2), (10, 3), (11, 4), (12, 5), (17,6), (21, 7), (22, 8), (23, 9)])
#session.setScanRunMatches([(4, 0)])

In [ ]:
session.displayScanRunMapping()

---

Save the raw session to database

---

In [ ]:
session.saveRawSession()

---

Help on commands

---

In [ ]:
# Get a list of all actions within an action module
#pglActionsFixes.help()
#pglActions.help()
#pglActionsMNE.help()
pglActionsMRI.help()

In [ ]:
# Get specific help on a paritcular action
help(pglActionsMRI.mriEstimateMotionAFNI)

---

Database

---

In [ ]:
# display database
pglDatabase.print(experimentName='nsd-mini')
#pglDatabase.print()

---

Load a session

---

In [ ]:
# load a specific session
#session = pglDatabase.loadSession(126)
#session = pglDatabase.loadSession(128)
session = pglDatabase.loadSession(136)

---

Display information about a session


---

In [ ]:
# display the top-level information
session.print()

In [ ]:
# more detailed trial-by-trial information, for a particular run
session.print(runNum=0, verbose=True)

In [ ]:
# display mri scans
session.mri.print()

In [ ]:
# display behavioral plots
session.display();

---

Show history of actions

---

In [ ]:
session.history()

In [ ]:
# show more detailed information
session.history(verbose=True)

In [ ]:
# show what products actions produced
session.history(showProducts=True)

In [ ]:
# show the logs for a particular action
session.history(3,showLogs=True)

In [ ]:
# get the products for a particular action and print
motion = session.getProduct("motion")
motion.print()

In [ ]:
# dispaly any plots related to the product
motion.display();

---

AFNI: check availability and optional install

---

In [ ]:
# Install AFNI. If already installed, will not try to re-install but will display information about installation
afni = pglAFNI()
afni.install()

In [ ]:
# check availability of specifc commands
afni = pglAFNI(directory="~/proj/afni/abin")
available = afni.checkAvailable("3dvolreg")

---

Run motion comp

---

In [ ]:
newSession = pglActionsMRI.mriEstimateMotionAFNI().execute(session=session, deferApply=True)
if newSession: session = newSession

In [ ]:
# display matrices
motion = session.getProduct('motion')
if motion: motion.display()

In [ ]:
# display motion correction estimates
session.mri.

In [ ]:
session.saveSession()

In [ ]:
session.print(verbose=True)

In [ ]:
newSession = pglActionsFixes.reconstructVolumeTriggersFromNifti.execute(session=session)
if newSession: session = newSession

In [ ]:
newSession.print()

In [ ]:
session.runs[0].tasks[0].stimulusDatabase.print()

In [ ]:
session.runs[0].getTrialsByStimulus()

In [ ]:
session.saveSession(checkpointName="within motion correction")

In [ ]:
from pgl.pglExternalTools import pglExternalTool, pglAFNI

pglAFNI.configure()

In [ ]:
from pgl import pglExternalTool
reports = pglExternalTool.print()

In [ ]:
from pgl.pglExternalTools import pglMATLAB

reports = pglMATLAB.print(validate=True, timeout=120)

In [ ]:
from pgl.pglExternalTools import pglGLMsingle

pglGLMsingle.configure()

In [ ]:
#session.setScanRunMatches([(4, 0)])
session.displayScanRunMapping()

In [ ]:
from pgl.pglActionsMRI import pglActionsMRI
from pgl.pglDatabase import pglDatabase

#session = pglDatabase.loadSession(136)
#session.setScanRunMatches([(4, 0), (5, 1)])

newSession = pglActionsMRI.GLMsingle.execute(session=session, model="B", stimDur=2.0, outputName="GLMsingleTypeBTest")
if newSession: session = newSession

In [ ]:
session.history()

In [ ]:
session.history(showProducts=True)
product = session.getProduct(4)

In [ ]:
product.display()

In [ ]:
from pgl.pglActionsMRI import pglActionsMRI
from pgl.pglDatabase import pglDatabase

session = pglDatabase.loadSession(136)
session.setScanRunMatches([(4, 0)])

session = pglActionsMRI.GLMsingle.execute(session=session,model="B", scanIndices=[4], stimDur=2.0, outputName="GLMsingleTypeBTest")

In [ ]:
from pathlib import Path
import re

# Change this if your checkout is somewhere else.
repoRoot = Path("~/proj/pgl").expanduser().resolve()
if not repoRoot.is_dir(): raise FileNotFoundError(f"Repository not found: {repoRoot}")

# Skip environments, generated files, and third-party checkouts.
skipDirs = {".git", ".venv", "venv", "env", "envs", "__pycache__", "build", "dist", "node_modules", ".ipynb_checkpoints", "site-packages"}
searchGroups = {
    "Session loading, saving, and dependencies": [
        r"\bclass\s+pglSession\b", r"\bdef\s+loadSession\b", r"\bdef\s+saveSession\b",
        r"\bparentSessionID\b", r"\brawSessionID\b", r"\bdependentSessions\b",
    ],
    "Dialogs and installation/configuration": [
        r"\bpglDialog\b", r"\bdef\s+\w*(?:configure|installation)\w*\b",
    ],
    "Migrations and schema versions": [
        r"\bmigration[s]?\b", r"\balembic\b", r"\bschemaVersion\b",
        r"\bschema_version\b", r"\buser_version\b",
    ],
    "Database selection and connections": [
        r"\bdatabaseName\b", r"\bpsycopg\b", r"\bcreate_engine\b",
        r"\bclass\s+\w*Database\w*\b",
    ],
}

patterns = {group: re.compile("|".join(expressions), re.IGNORECASE) for group, expressions in searchGroups.items()}
searchResults = {group: [] for group in searchGroups}
fileHints = []

for path in sorted(repoRoot.rglob("*")):
    relativePath = path.relative_to(repoRoot)
    if any(part in skipDirs or part.endswith(".egg-info") for part in relativePath.parts): continue
    if not path.is_file() or path.suffix.lower() not in {".py", ".sql", ".toml", ".ini", ".md"}: continue
    if path.stat().st_size > 1_000_000: continue

    if re.search(r"session|dialog|migrat|schema|database|install|config", str(relativePath), re.IGNORECASE):
        fileHints.append(str(relativePath))

    try:
        lines = path.read_text(encoding="utf-8").splitlines()
    except (OSError, UnicodeError):
        continue

    for group, pattern in patterns.items():
        lineNumbers = [number for number, line in enumerate(lines, 1) if pattern.search(line)]
        if lineNumbers: searchResults[group].append((str(relativePath), lineNumbers))

print(f"Repository: {repoRoot}")

for group, matches in searchResults.items():
    print(f"\n{group}")
    print("-" * len(group))
    for fileName, lineNumbers in matches[:30]:
        locations = ", ".join(map(str, lineNumbers[:10]))
        suffix = " ..." if len(lineNumbers) > 10 else ""
        print(f"  {fileName}  [lines {locations}{suffix}]")
    if not matches: print("  No matches.")
    if len(matches) > 30: print(f"  ... {len(matches) - 30} more files; available in searchResults.")

print("\nAdditional filename clues")
for fileName in fileHints[:60]: print(f"  {fileName}")
if len(fileHints) > 60: print(f"  ... {len(fileHints) - 60} more; available in fileHints.")

In [ ]:
import ast
import re
from pathlib import Path

repoRoot = Path("~/proj/pgl").expanduser().resolve()
outputPath = Path.home() / "pglExportStorageReview.txt"
if not repoRoot.is_dir(): raise FileNotFoundError(f"Repository not found: {repoRoot}")

methodNames = {
    "_prepareCheckpointBaseline",
    "_getCheckpointHistoryBoundary",
    "_getConfiguredStorageBackend",
    "getOrCreateUser",
    "createRawCheckpoint",
    "createDerivedCheckpoint",
    "_completeSessionBundleCheckpoint",
    "_publishSessionMatrices",
    "_openRegisteredMatrix",
    "_grantApplicationAccess",
    "validateRawSessionManifest",
    "validateRawSessionV2Manifest",
    "validateDerivedSessionManifest",
    "_validateSessionBundleManifest",
}

sections = []
foundNames = set()

def appendSource(relativePath, lines, node, label):
    # Include decorators so extracted methods retain their calling convention.
    startLine = min([node.lineno] + [decorator.lineno for decorator in getattr(node, "decorator_list", [])])
    content = "\n".join(f"{number:5}: {lines[number - 1]}" for number in range(startLine, node.end_lineno + 1))
    sections.append(f"\n{'=' * 80}\n{relativePath}: {label}\n{'=' * 80}\n{content}\n")

relativePath = "pgl/pglDatabase.py"
source = (repoRoot / relativePath).read_text(encoding="utf-8")
lines = source.splitlines()
tree = ast.parse(source)

for classNode in tree.body:
    if not isinstance(classNode, ast.ClassDef): continue

    # Include the existing payload-access context in full.
    if classNode.name == "pglDatabaseAccess":
        appendSource(relativePath, lines, classNode, classNode.name)

    if classNode.name != "pglDatabase": continue
    for node in classNode.body:
        if not isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)): continue
        if node.name not in methodNames: continue
        appendSource(relativePath, lines, node, f"{classNode.name}.{node.name}")
        foundNames.add(node.name)

# Index related helpers so we can reuse their actual APIs rather than guess.
namePattern = re.compile(r"copy|verif|hash|manifest|matrix|register|backend|location|storage", re.IGNORECASE)

for relativePath in ("pgl/pglStorage.py", "pgl/pglDatabase.py"):
    source = (repoRoot / relativePath).read_text(encoding="utf-8")
    tree = ast.parse(source)
    entries = []

    for classNode in tree.body:
        if not isinstance(classNode, ast.ClassDef): continue
        for node in classNode.body:
            if not isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)): continue
            if not namePattern.search(node.name): continue
            entries.append(f"{classNode.name}.{node.name}({ast.unparse(node.args)}) [lines {node.lineno}-{node.end_lineno}]")

    sections.append(f"\n--- Helper index: {relativePath} ---\n" + "\n".join(entries) + "\n")

missingNames = methodNames - foundNames
if missingNames:
    sections.append("\nRequested methods not found:\n" + "\n".join(sorted(missingNames)) + "\n")

outputPath.write_text("".join(sections), encoding="utf-8")
print(f"Created: {outputPath}")
print(f"Size: {outputPath.stat().st_size:,} bytes")
if missingNames: print("Methods not found:", ", ".join(sorted(missingNames)))

In [ ]:
from pathlib import Path

repoRoot = Path("~/proj/pgl").expanduser().resolve()
if not repoRoot.is_dir(): raise FileNotFoundError(f"Repository not found: {repoRoot}")

skipDirs = {".git", ".venv", "venv", "env", "envs", "__pycache__", "build", "dist", "node_modules", "site-packages"}
testFiles = []

for path in sorted(repoRoot.rglob("*.py")):
    relativePath = path.relative_to(repoRoot)
    if any(part in skipDirs or part.endswith(".egg-info") for part in relativePath.parts): continue
    if path.name == "conftest.py" or path.name.startswith("test_") or path.name.endswith("_test.py"):
        testFiles.append(str(relativePath))

print("Test files:")
for relativePath in testFiles: print(f"  {relativePath}")
if not testFiles: print("  No conventional Python test files found.")

print("\nTest configuration files:")
for fileName in ("pyproject.toml", "pytest.ini", "setup.cfg", "tox.ini"):
    if (repoRoot / fileName).is_file(): print(f"  {fileName}")

print("\nCurrent migrations:")
migrationDir = repoRoot / "database" / "migrations"
for path in sorted(migrationDir.glob("*.sql")): print(f"  {path.name}")
if not migrationDir.is_dir(): print("  Migration directory not found.")

In [ ]:
from pathlib import Path
import os

repoRoot = Path("~/proj/pgl").expanduser().resolve()
if not repoRoot.is_dir(): raise FileNotFoundError(f"Repository not found: {repoRoot}")

skipDirs = {".git", ".venv", "venv", "env", "envs", "__pycache__", "build", "dist", "node_modules", "site-packages", ".ipynb_checkpoints"}
testFiles = []

# Prune excluded directories rather than traversing installed environments.
for directory, dirNames, fileNames in os.walk(repoRoot):
    dirNames[:] = sorted(name for name in dirNames if name not in skipDirs and not name.endswith(".egg-info"))
    for fileName in sorted(fileNames):
        if fileName == "conftest.py" or (fileName.endswith(".py") and (fileName.startswith("test_") or fileName.endswith("_test.py"))):
            testFiles.append(str((Path(directory) / fileName).relative_to(repoRoot)))

print("TEST FILES")
for relativePath in sorted(testFiles): print(f"  {relativePath}")
if not testFiles: print("  No conventional Python test files found.")

print("\nTEST CONFIGURATION FILES")
for fileName in ("pyproject.toml", "pytest.ini", "setup.cfg", "tox.ini"):
    if (repoRoot / fileName).is_file(): print(f"  {fileName}")

print("\nCURRENT MIGRATIONS")
migrationDir = repoRoot / "database" / "migrations"
for path in sorted(migrationDir.glob("*.sql")): print(f"  {path.name}")
if not migrationDir.is_dir(): print("  Migration directory not found.")

In [ ]:
from pathlib import Path
from datetime import datetime
import ast
import shutil
import textwrap

repoPath = Path("~/proj/pgl").expanduser().resolve()
databasePath = repoPath / "pgl" / "pglDatabase.py"
sessionPath = repoPath / "pgl" / "pglSession.py"
migrationPath = repoPath / "database" / "migrations" / "0015_schema_identity.sql"

if not migrationPath.is_file(): raise FileNotFoundError("Save 0015_schema_identity.sql before applying this update.")

databaseSource = databasePath.read_text(encoding="utf-8")
sessionSource = sessionPath.read_text(encoding="utf-8")

newHelpers = r'''
    @classmethod
    def _requireStableDatabaseIdentity(cls, connection, postgres):
        """Require current migration history, then read the persistent UUID.

        Uses the caller's transaction. No migrations, grants, or schema writes.
        SQL access failures retain their original exception categories.

        Public load/save coordinators call this before accessing checkpoint
        payloads or performing publication writes.
        """
        report = cls._inspectMigrationCompatibility(connection, postgres)

        if not report["compatible"]:
            details = "\n".join(issue["message"] for issue in report["issues"])
            raise RuntimeError(
                "Scientific database migration compatibility could not be established.\n"
                + details
            )

        return cls._readStableDatabaseID(connection, postgres)

    @staticmethod
    def _validateCheckpointBinding(binding):
        """Validate runtime binding metadata without reading settings or SQL.

        This record identifies the selected saved checkpoint, not the complete
        current scientific state. Unsaved working changes may exist.

        Runtime endpoint checks remain distinct from durable provenance.
        Returns an independent copy; never upgrades a legacy history record.
        """
        expectedFields = {"version", "checkpoint", "databaseIdentity"}

        if type(binding) is not dict or set(binding) != expectedFields:
            raise ValueError("Invalid saved-checkpoint binding fields.")
        if type(binding["version"]) is not int or binding["version"] != 1:
            raise ValueError("Unsupported saved-checkpoint binding version.")

        reference = binding["checkpoint"]
        if type(reference) is not dict or set(reference) != {"databaseID", "sessionID", "databaseName"}:
            raise ValueError("Invalid database-qualified checkpoint reference.")

        sessionID = reference["sessionID"]
        if type(sessionID) is not int or not 0 <= sessionID <= 9223372036854775807:
            raise ValueError("Bound sessionID must be a nonnegative PostgreSQL bigint.")

        databaseID = reference["databaseID"]
        if not isinstance(databaseID, str):
            raise ValueError("Bound databaseID must be a canonical UUID string.")

        try:
            parsedID = uuid.UUID(databaseID)
        except (ValueError, AttributeError):
            raise ValueError("Bound databaseID is invalid.") from None

        if str(parsedID) != databaseID or parsedID.int == 0:
            raise ValueError("Bound databaseID must be a canonical nonzero UUID.")

        databaseName = reference["databaseName"]
        if (
            not isinstance(databaseName, str) or not databaseName.strip()
            or any(character in databaseName for character in ("\x00", "\n", "\r"))
        ):
            raise ValueError("Bound databaseName must be a nonblank profile label without NUL or newlines.")

        identity = binding["databaseIdentity"]
        if type(identity) is not dict:
            raise ValueError("Bound runtime database identity must be a dictionary.")

        fields = {"transport", "host", "port", "database", "schema"}
        if identity.get("transport") == "ssh":
            fields.update({"sshHost", "sshPort", "sshUser"})
        elif identity.get("transport") != "direct":
            raise ValueError("Unsupported bound database transport.")

        if set(identity) != fields:
            raise ValueError("Invalid bound runtime database identity fields.")

        for name in ("host", "database", "schema"):
            if not isinstance(identity[name], str) or not identity[name].strip() or "\x00" in identity[name]:
                raise ValueError(f"Invalid bound database endpoint field: {name}.")

        for name in ("port", "sshPort"):
            if name in identity and (type(identity[name]) is not int or not 1 <= identity[name] <= 65535):
                raise ValueError(f"Invalid bound database endpoint port: {name}.")

        if identity["transport"] == "ssh":
            if not isinstance(identity["sshHost"], str) or not identity["sshHost"].strip() or "\x00" in identity["sshHost"]:
                raise ValueError("Invalid bound SSH host.")
            if not isinstance(identity["sshUser"], str) or "\x00" in identity["sshUser"]:
                raise ValueError("Invalid bound SSH user.")

        return deepcopy(binding)

    @classmethod
    def _prepareCheckpointBinding(cls, sessionID, postgres, *, databaseID, databaseName):
        """Prepare a binding without modifying a working session.

        databaseID must have been read from the caller's verified SQL snapshot.
        databaseName is the resolved PGL profile label, not the SQL database name.

        Saves prepare this before commit and install it after cleanup.
        No scientific history boundary is inferred, including for raw saves.
        """
        binding = {
            "version": 1,
            "checkpoint": {
                "databaseID": databaseID,
                "sessionID": sessionID,
                "databaseName": databaseName,
            },
            "databaseIdentity": cls._checkpointDatabaseIdentity(postgres),
        }
        return cls._validateCheckpointBinding(binding)

    @classmethod
    def _getBoundCheckpointReference(cls, session):
        """Return the selected saved-checkpoint reference, or None if unbound.

        Reads runtime bookkeeping only. Never consults the current default
        database, opens payloads, or changes history.

        A numeric sessionID without this binding does not establish a source
        database. Existing pre-update working sessions should be reloaded.

        A reference identifies the saved checkpoint beneath a working object;
        it is not a claim that every current scientific value was saved there.
        """
        binding = getattr(session, "_checkpointBinding", {})

        if type(binding) is not dict:
            raise ValueError("Invalid saved-checkpoint binding.")
        if not binding: return None

        validated = cls._validateCheckpointBinding(binding)
        sessionID = getattr(session, "sessionID", None)

        if type(sessionID) is not int or sessionID != validated["checkpoint"]["sessionID"]:
            raise ValueError(
                "Working sessionID differs from its saved-checkpoint binding. "
                "Reload the intended checkpoint rather than manually retargeting its ID."
            )

        return validated["checkpoint"]
'''

def getClass(source, className):
    tree = ast.parse(source)
    matches = [node for node in tree.body if isinstance(node, ast.ClassDef) and node.name == className]
    if len(matches) != 1: raise ValueError(f"Expected exactly one class {className}.")
    return matches[0]

def getMethodText(source, className, methodName):
    classNode = getClass(source, className)
    matches = [node for node in classNode.body if isinstance(node, ast.FunctionDef) and node.name == methodName]
    if len(matches) != 1: raise ValueError(f"Expected exactly one {className}.{methodName}.")
    node = matches[0]
    startLine = min([node.lineno] + [decorator.lineno for decorator in node.decorator_list])
    lines = source.splitlines(keepends=True)
    return "".join(lines[startLine - 1:node.end_lineno]), startLine - 1, node.end_lineno

def replaceOnce(source, old, new, label):
    count = source.count(old)
    if count != 1: raise ValueError(f"{label}: expected one source match, found {count}; no files written.")
    return source.replace(old, new, 1)

def updateMethod(source, className, methodName, replacements):
    methodText, start, stop = getMethodText(source, className, methodName)
    for old, new in replacements:
        methodText = replaceOnce(methodText, old, new, f"{className}.{methodName}")

    # Verify each complete resulting function before placing it in the module.
    ast.parse(textwrap.dedent(methodText))
    lines = source.splitlines(keepends=True)
    return "".join(lines[:start]) + methodText + "".join(lines[stop:])

helperNames = {
    "_requireStableDatabaseIdentity",
    "_validateCheckpointBinding",
    "_prepareCheckpointBinding",
    "_getBoundCheckpointReference",
}
existingNames = {
    node.name for node in getClass(databaseSource, "pglDatabase").body
    if isinstance(node, ast.FunctionDef)
}
if helperNames.intersection(existingNames): raise ValueError("Some binding helpers already exist; refusing to apply the update twice.")
if not {"_inspectMigrationCompatibility", "_readStableDatabaseID"}.issubset(existingNames):
    raise ValueError("The previous migration-inspection update is missing.")

# Prepare all source changes in memory before creating backups or writing.
updatedDatabase = databaseSource
updatedSession = sessionSource

updatedSession = replaceOnce(
    updatedSession,
    '    sessionID = Int(default_value=None, allow_none=True, help="Most recently loaded or successfully saved database checkpoint ID", serialize=False)\n',
    '    sessionID = Int(default_value=None, allow_none=True, help="Most recently loaded or successfully saved database checkpoint ID", serialize=False)\n'
    '    # Saved source identity is separate from the history baseline: raw saves\n'
    '    # bind a checkpoint without asserting a new working-history boundary.\n'
    '    _checkpointBinding = Dict(default_value={}, help="Storage-managed saved-checkpoint identity; not a dirty-state detector").tag(serialize=False)\n',
    "pglSession checkpoint binding trait",
)

# Load: obtain identity in the same read-only snapshot, and retain only a
# private configuration copy—not a SQL connection—for future source access.
updatedDatabase = updateMethod(updatedDatabase, "pglDatabase", "loadSession", [
    (
        '                    connection.execute("SET TRANSACTION ISOLATION LEVEL REPEATABLE READ, READ ONLY")\n',
        '                    connection.execute("SET TRANSACTION ISOLATION LEVEL REPEATABLE READ, READ ONLY")\n'
        '                    databaseID = cls._requireStableDatabaseIdentity(connection, postgres)\n',
    ),
    (
        '                        session = loader(connection, postgres, sessionID, **loadOptions)\n',
        '                        session = loader(connection, postgres, sessionID, **loadOptions)\n'
        '                        pendingBinding = cls._prepareCheckpointBinding(sessionID, postgres, databaseID=databaseID, databaseName=resolvedSettings.name)\n'
        '                        pendingPostgres = postgres.withTimeouts()\n',
    ),
    (
        '            cls._bindCheckpointBaseline(session, sessionID, postgres)\n',
        '            cls._bindCheckpointBaseline(session, sessionID, postgres)\n'
        '            session._checkpointBinding = pendingBinding\n'
        '            session._checkpointPostgres = pendingPostgres\n',
    ),
    (
        '        Does not publish or modify database records.\n',
        '        Requires current migration history and a persistent schema identity.\n'
        '        Does not apply migrations, publish, or modify database records.\n',
    ),
])

# Derived save: retain existing endpoint checks and also compare the UUID
# actually read from SQL, protecting against replacement at the same endpoint.
updatedDatabase = updateMethod(updatedDatabase, "pglDatabase", "saveSession", [
    (
        '        originalBaseline = deepcopy(session._checkpointBaseline)\n',
        '        originalBaseline = deepcopy(session._checkpointBaseline)\n'
        '        originalReference = cls._getBoundCheckpointReference(session)\n'
        '        originalBinding = deepcopy(session._checkpointBinding)\n',
    ),
    (
        '                if backendID is None and not postgres.storageLocations:\n',
        '                if originalBinding and originalBinding["databaseIdentity"] != cls._checkpointDatabaseIdentity(postgres):\n'
        '                    raise ValueError("The selected database differs from this session\'s saved-checkpoint endpoint.")\n\n'
        '                if backendID is None and not postgres.storageLocations:\n',
    ),
    (
        '                        connection.execute("SET TRANSACTION ISOLATION LEVEL READ COMMITTED")\n',
        '                        connection.execute("SET TRANSACTION ISOLATION LEVEL READ COMMITTED")\n'
        '                        databaseID = cls._requireStableDatabaseIdentity(connection, postgres)\n'
        '                        if originalReference is not None and originalReference["databaseID"] != databaseID:\n'
        '                            raise ValueError("The database UUID differs from this session\'s saved-checkpoint identity.")\n',
    ),
    (
        '                        pendingBaseline = cls._prepareCheckpointBaseline(session, sessionID, postgres)\n',
        '                        if session._checkpointBinding != originalBinding:\n'
        '                            raise ValueError("Working saved-checkpoint binding changed while saving.")\n\n'
        '                        pendingBaseline = cls._prepareCheckpointBaseline(session, sessionID, postgres)\n'
        '                        pendingBinding = cls._prepareCheckpointBinding(sessionID, postgres, databaseID=databaseID, databaseName=resolvedSettings.name)\n'
        '                        pendingPostgres = postgres.withTimeouts()\n',
    ),
    (
        '                session._checkpointBaseline = pendingBaseline\n',
        '                session._checkpointBaseline = pendingBaseline\n'
        '                session._checkpointBinding = pendingBinding\n'
        '                session._checkpointPostgres = pendingPostgres\n',
    ),
    (
        '        Payload writes are not transactional.\n',
        '        Payload writes are not transactional.\n'
        '        Requires current migrations; never applies migrations automatically.\n',
    ),
])

# Raw save: bind the completed acquisition checkpoint without assigning a
# history baseline. The database coordinator now owns the sessionID update.
updatedDatabase = updateMethod(updatedDatabase, "pglDatabase", "saveRawSession", [
    (
        '        sessionID = None\n',
        '        originalReference = cls._getBoundCheckpointReference(session)\n'
        '        originalBinding = deepcopy(session._checkpointBinding)\n'
        '        originalSessionID = session.sessionID\n'
        '        originalBaseline = deepcopy(session._checkpointBaseline)\n\n'
        '        sessionID = None\n',
    ),
    (
        '                if backendID is None and not postgres.storageLocations:\n',
        '                if originalBinding and originalBinding["databaseIdentity"] != cls._checkpointDatabaseIdentity(postgres):\n'
        '                    raise ValueError("The selected database differs from this session\'s saved-checkpoint endpoint.")\n\n'
        '                if backendID is None and not postgres.storageLocations:\n',
    ),
    (
        '                    with connection.transaction():\n',
        '                    with connection.transaction():\n'
        '                        databaseID = cls._requireStableDatabaseIdentity(connection, postgres)\n'
        '                        if originalReference is not None and originalReference["databaseID"] != databaseID:\n'
        '                            raise ValueError("The database UUID differs from this session\'s saved-checkpoint identity.")\n\n',
    ),
    (
        '                        operation = "committing the checkpoint transaction"\n',
        '                        # Prepare binding before commit; installation on the\n'
        '                        # working object waits for connection cleanup below.\n'
        '                        if session.sessionID != originalSessionID or session._checkpointBinding != originalBinding or session._checkpointBaseline != originalBaseline:\n'
        '                            raise ValueError("Working checkpoint bookkeeping changed while saving raw acquisition contents.")\n'
        '                        pendingBinding = cls._prepareCheckpointBinding(sessionID, postgres, databaseID=databaseID, databaseName=resolvedSettings.name)\n'
        '                        pendingPostgres = postgres.withTimeouts()\n\n'
        '                        operation = "committing the checkpoint transaction"\n',
    ),
    (
        '                progress.finish(f"Raw checkpoint {sessionID} saved")\n',
        '                # Raw publication identifies an acquisition checkpoint,\n'
        '                # not a new history boundary for the current working state.\n'
        '                session.sessionID = sessionID\n'
        '                session._checkpointBinding = pendingBinding\n'
        '                session._checkpointPostgres = pendingPostgres\n\n'
        '                progress.finish(f"Raw checkpoint {sessionID} saved")\n',
    ),
    (
        '        No automatic retry, connection termination, or payload deletion occurs.\n',
        '        No automatic retry, connection termination, or payload deletion occurs.\n\n'
        '        Requires current migrations; never applies them automatically.\n'
        '        Binds sessionID and saved source identity after connection cleanup,\n'
        '        without establishing or replacing a working-history baseline.\n',
    ),
])

updatedSession = updateMethod(updatedSession, "pglSession", "saveRawSession", [
    (
        '        Assigns self.sessionID only after successful publication and cleanup.\n'
        '        Does not bind a new history baseline or clear action history.\n',
        '        The database coordinator binds self.sessionID and saved source identity\n'
        '        only after successful publication and connection cleanup.\n'
        '        Does not bind a new history baseline or clear action history.\n',
    ),
    (
        '        if sessionID is not None:\n'
        '            self.sessionID = sessionID\n\n',
        '        # Identity assignment belongs to the database coordinator, which\n'
        '        # has the verified schema UUID and resolved source configuration.\n',
    ),
])

# Insert complete new methods at the end of the existing database class.
classNode = getClass(updatedDatabase, "pglDatabase")
lines = updatedDatabase.splitlines(keepends=True)
updatedDatabase = "".join(lines[:classNode.end_lineno]) + "\n" + newHelpers + "\n" + "".join(lines[classNode.end_lineno:])

ast.parse(updatedDatabase, filename=str(databasePath))
ast.parse(updatedSession, filename=str(sessionPath))

# Refuse detectable concurrent edits before writing.
if databasePath.read_text(encoding="utf-8") != databaseSource: raise RuntimeError("pglDatabase.py changed during preparation.")
if sessionPath.read_text(encoding="utf-8") != sessionSource: raise RuntimeError("pglSession.py changed during preparation.")

backupPath = Path.home() / f"pglIdentityBindingBackup_{datetime.now().strftime('%Y%m%d_%H%M%S_%f')}"
backupPath.mkdir()
shutil.copy2(databasePath, backupPath / databasePath.name)
shutil.copy2(sessionPath, backupPath / sessionPath.name)

# These two filesystem writes are not a multi-file transaction.
# Original source is retained in backupPath if either write fails.
databasePath.write_text(updatedDatabase, encoding="utf-8")
sessionPath.write_text(updatedSession, encoding="utf-8")

print("Updated pglDatabase.py and pglSession.py; Python syntax checks passed.")
print(f"Original source backups: {backupPath}")
print("No database connections, migrations, or payload operations were executed.")

In [ ]:
from pathlib import Path
from types import SimpleNamespace
from copy import deepcopy
import ast
import unittest
import uuid

repoPath = Path("~/proj/pgl").expanduser().resolve()
sourcePath = repoPath / "pgl" / "pglDatabase.py"
sourceTree = ast.parse(sourcePath.read_text(encoding="utf-8"))
databaseNode = next(node for node in sourceTree.body if isinstance(node, ast.ClassDef) and node.name == "pglDatabase")

helperNames = {
    "_checkpointDatabaseIdentity",
    "_requireStableDatabaseIdentity",
    "_validateCheckpointBinding",
    "_prepareCheckpointBinding",
    "_getBoundCheckpointReference",
}
helperNodes = [
    deepcopy(node) for node in databaseNode.body
    if isinstance(node, ast.FunctionDef) and node.name in helperNames
]
if {node.name for node in helperNodes} != helperNames: raise ValueError("Binding helpers are missing.")

testClassNode = ast.ClassDef(name="BindingHelpers", bases=[], keywords=[], body=helperNodes, decorator_list=[])
testModule = ast.fix_missing_locations(ast.Module(body=[testClassNode], type_ignores=[]))
namespace = {"uuid": uuid, "deepcopy": deepcopy}
exec(compile(testModule, str(sourcePath), "exec"), namespace)
bindingHelpers = namespace["BindingHelpers"]

class CheckpointBindingTests(unittest.TestCase):
    def setUp(self):
        self.databaseID = "8d4dcf5a-4d74-42ed-a7dd-d70e70308c92"
        self.otherDatabaseID = "067ec061-c8cb-4b41-945c-315dc7af5fa4"
        self.postgres = SimpleNamespace(
            connectionTransport="direct",
            connectionHost="127.0.0.1",
            connectionPort=5432,
            databaseName="actual_sql_database",
            databaseSchema="scientific",
        )

    def prepare(self, sessionID=0, *, databaseID=None, databaseName="sourceProfile"):
        return bindingHelpers._prepareCheckpointBinding(
            sessionID,
            self.postgres,
            databaseID=self.databaseID if databaseID is None else databaseID,
            databaseName=databaseName,
        )

    def testSessionZeroAndProfileLabel(self):
        binding = self.prepare()
        self.assertEqual(binding["checkpoint"], {
            "databaseID": self.databaseID,
            "sessionID": 0,
            "databaseName": "sourceProfile",
        })
        self.assertEqual(binding["databaseIdentity"]["database"], "actual_sql_database")

    def testInvalidSessionIDs(self):
        for value in (True, False, -1, 2**63, 1.0, "1", None):
            with self.subTest(value=value):
                with self.assertRaises(ValueError): self.prepare(value)

    def testMaximumSessionID(self):
        self.assertEqual(self.prepare(2**63 - 1)["checkpoint"]["sessionID"], 2**63 - 1)

    def testInvalidDatabaseIDs(self):
        for value in ("invalid", self.databaseID.upper(), str(uuid.UUID(int=0)), 42):
            with self.subTest(value=value):
                with self.assertRaises(ValueError): self.prepare(databaseID=value)

    def testInvalidProfileLabels(self):
        for value in ("", " ", "source\nprofile", "source\x00profile", None):
            with self.subTest(value=value):
                with self.assertRaises(ValueError): self.prepare(databaseName=value)

    def testUnsavedInput(self):
        session = SimpleNamespace(sessionID=None, _checkpointBinding={})
        self.assertIsNone(bindingHelpers._getBoundCheckpointReference(session))

    def testLegacyNumericIDDoesNotInferDatabase(self):
        session = SimpleNamespace(sessionID=42)
        self.assertIsNone(bindingHelpers._getBoundCheckpointReference(session))

    def testBoundReference(self):
        session = SimpleNamespace(sessionID=0, _checkpointBinding=self.prepare())
        self.assertEqual(bindingHelpers._getBoundCheckpointReference(session), session._checkpointBinding["checkpoint"])

    def testManualIDChangeRejected(self):
        for value in (1, None, False):
            with self.subTest(value=value):
                session = SimpleNamespace(sessionID=value, _checkpointBinding=self.prepare())
                with self.assertRaises(ValueError):
                    bindingHelpers._getBoundCheckpointReference(session)

    def testMalformedBindingRejected(self):
        for binding in (None, [], {"version": 1}, {"version": True}):
            with self.subTest(binding=binding):
                session = SimpleNamespace(sessionID=0, _checkpointBinding=binding)
                with self.assertRaises(ValueError):
                    bindingHelpers._getBoundCheckpointReference(session)

    def testReferenceIsIndependent(self):
        session = SimpleNamespace(sessionID=0, _checkpointBinding=self.prepare())
        reference = bindingHelpers._getBoundCheckpointReference(session)
        reference["databaseName"] = "changed"
        self.assertEqual(session._checkpointBinding["checkpoint"]["databaseName"], "sourceProfile")

    def testValidationReturnsIndependentCopy(self):
        original = self.prepare()
        validated = bindingHelpers._validateCheckpointBinding(original)
        validated["databaseIdentity"]["host"] = "changed"
        self.assertEqual(original["databaseIdentity"]["host"], "127.0.0.1")

    def testSameNumericIDDifferentDatabases(self):
        left = self.prepare(42)
        right = self.prepare(42, databaseID=self.otherDatabaseID)
        self.assertNotEqual(left["checkpoint"], right["checkpoint"])

    def testAlternateEndpointsRetainStableID(self):
        local = self.prepare(42)
        self.postgres.connectionHost = "alternate.example"
        alternate = self.prepare(42, databaseName="alternateProfile")
        self.assertEqual(local["checkpoint"]["databaseID"], alternate["checkpoint"]["databaseID"])
        self.assertNotEqual(local["databaseIdentity"], alternate["databaseIdentity"])

    def testSSHIdentityRetainedSeparately(self):
        self.postgres.connectionTransport = "ssh"
        self.postgres.sshHost = "installation"
        self.postgres.sshPort = 22
        self.postgres.sshUser = ""
        binding = self.prepare()
        self.assertEqual(binding["databaseIdentity"]["sshHost"], "installation")
        self.assertNotIn("sshHost", binding["checkpoint"])

    def testHistoryAndBaselineUntouched(self):
        history = [{"sessionID": 42}]
        baseline = {"version": 1, "historyCount": 0}
        session = SimpleNamespace(
            sessionID=0,
            _checkpointBinding=self.prepare(),
            actionHistory=history,
            _checkpointBaseline=baseline,
        )
        bindingHelpers._getBoundCheckpointReference(session)
        self.assertIs(session.actionHistory, history)
        self.assertIs(session._checkpointBaseline, baseline)
        self.assertEqual(history, [{"sessionID": 42}])

    def testCompatibilityFailureDoesNotReadIdentity(self):
        class Incompatible(bindingHelpers):
            @classmethod
            def _inspectMigrationCompatibility(cls, connection, postgres):
                return {"compatible": False, "issues": [{"message": "Pending migrations: initialize on the installation machine."}]}

            @classmethod
            def _readStableDatabaseID(cls, connection, postgres):
                raise AssertionError("Identity must not be read after failed compatibility.")

        with self.assertRaisesRegex(RuntimeError, "installation machine"):
            Incompatible._requireStableDatabaseIdentity(object(), self.postgres)

    def testCompatibilitySuccessReadsIdentity(self):
        expectedID = self.databaseID

        class Compatible(bindingHelpers):
            @classmethod
            def _inspectMigrationCompatibility(cls, connection, postgres):
                return {"compatible": True, "issues": []}

            @classmethod
            def _readStableDatabaseID(cls, connection, postgres):
                return expectedID

        self.assertEqual(Compatible._requireStableDatabaseIdentity(object(), self.postgres), expectedID)

    def testInspectionErrorsPropagate(self):
        class Inaccessible(bindingHelpers):
            @classmethod
            def _inspectMigrationCompatibility(cls, connection, postgres):
                raise PermissionError("Migration history inaccessible.")

        with self.assertRaises(PermissionError):
            Inaccessible._requireStableDatabaseIdentity(object(), self.postgres)

suite = unittest.defaultTestLoader.loadTestsFromTestCase(CheckpointBindingTests)
testResult = unittest.TextTestRunner(verbosity=2).run(suite)
if not testResult.wasSuccessful(): raise AssertionError("Checkpoint binding unit tests failed.")

In [ ]:
from pathlib import Path
from types import SimpleNamespace
from copy import deepcopy
import ast
import unittest
import uuid

repoPath = Path("~/proj/pgl").expanduser().resolve()
sourcePath = repoPath / "pgl" / "pglDatabase.py"
sourceTree = ast.parse(sourcePath.read_text(encoding="utf-8"))
databaseNode = next(node for node in sourceTree.body if isinstance(node, ast.ClassDef) and node.name == "pglDatabase")

methodNames = {
    "saveSession",
    "saveRawSession",
    "_checkpointDatabaseIdentity",
    "_validateCheckpointBinding",
    "_prepareCheckpointBinding",
    "_getBoundCheckpointReference",
}
methodNodes = [
    deepcopy(node) for node in databaseNode.body
    if isinstance(node, ast.FunctionDef) and node.name in methodNames
]
if {node.name for node in methodNodes} != methodNames:
    raise ValueError("Expected save coordinators and binding helpers were not found.")

class RemovePGLImports(ast.NodeTransformer):
    """Replace only relative PGL imports with injected test collaborators."""

    def visit_ImportFrom(self, node):
        return None if node.level else node

methodNodes = [RemovePGLImports().visit(node) for node in methodNodes]
testClass = ast.ClassDef(name="SaveCoordinators", bases=[], keywords=[], body=methodNodes, decorator_list=[])
testModule = ast.fix_missing_locations(ast.Module(body=[testClass], type_ignores=[]))

# Real exception classes are used, but no connections are opened.
import psycopg

class TestMessages:
    @staticmethod
    def warning(*args, **kwargs): pass

    @staticmethod
    def message(*args, **kwargs): pass

class TestProgress:
    def __init__(self, *args, **kwargs):
        self._description = ""

    def __enter__(self): return self
    def __exit__(self, *args): return False
    def section(self, *args, **kwargs): return self

    def __call__(self, fraction, description):
        self._description = description

    def finish(self, description):
        self._description = description

namespace = {
    "deepcopy": deepcopy,
    "uuid": uuid,
    "psycopg": psycopg,
    "pglMessages": TestMessages,
    "pglProgress": TestProgress,
    "pglStorage": SimpleNamespace(requireSafeSave=lambda session: None),
}
exec(compile(testModule, str(sourcePath), "exec"), namespace)
saveCoordinators = namespace["SaveCoordinators"]

class SimulatedTransaction:
    """Expose commit ordering and simulate an uncertain commit."""

    def __init__(self, scenario):
        self.scenario = scenario

    def __enter__(self):
        self.scenario.events.append("transactionEntered")
        return self

    def __exit__(self, errorType, error, traceback):
        scenario = self.scenario
        scenario.requireOriginalBookkeeping()

        if errorType is not None:
            scenario.events.append("rollback")
            return False

        scenario.events.append("commitAttempted")
        if scenario.failure == "commit":
            raise ConnectionError("Simulated lost connection during commit.")

        scenario.events.append("commitConfirmed")
        return False

class SimulatedConnection:
    def __init__(self, scenario):
        self.scenario = scenario

    def transaction(self): return SimulatedTransaction(self.scenario)

    def execute(self, statement, parameters=None):
        # Publication itself is stubbed. Only transaction setup should reach here.
        if statement != "SET TRANSACTION ISOLATION LEVEL READ COMMITTED":
            raise AssertionError(f"Unexpected SQL in coordinator test: {statement}")
        self.scenario.events.append("isolationConfigured")
        return self

class SimulatedConnectionContext:
    def __init__(self, scenario):
        self.scenario = scenario

    def __enter__(self):
        self.scenario.events.append("connectionEntered")
        if self.scenario.failure == "cancel": return None
        return SimulatedConnection(self.scenario)

    def __exit__(self, errorType, error, traceback):
        scenario = self.scenario

        # Both ID and binding must still be unchanged when cleanup begins.
        scenario.requireOriginalBookkeeping()
        scenario.events.append("cleanupStarted")

        if scenario.failure == "cleanup" and errorType is None:
            raise ConnectionError("Simulated connection-cleanup failure.")

        scenario.events.append("cleanupFinished")
        return False

class SimulatedPostgres:
    def __init__(self, scenario):
        self.scenario = scenario
        self.connectionTransport = "direct"
        self.connectionHost = "127.0.0.1"
        self.connectionPort = 5432
        self.databaseName = "disposable_simulation"
        self.databaseSchema = "scientific"
        self.storageLocations = ["/not-accessed"]
        self.lockTimeoutSeconds = 30.0

    def connect(self, **kwargs): return SimulatedConnectionContext(self.scenario)

    def withTimeouts(self, **kwargs):
        # A distinct runtime configuration copy, with no actual connection.
        copied = SimulatedPostgres(self.scenario)
        copied.connectionHost = self.connectionHost
        return copied

class SaveScenario:
    def __init__(self, *, raw=False, failure=None, bound=True):
        self.raw = raw
        self.failure = failure
        self.events = []
        self.databaseID = "8d4dcf5a-4d74-42ed-a7dd-d70e70308c92"
        self.postgres = SimulatedPostgres(self)
        self.settings = SimpleNamespace(
            name="explicitProfile",
            username="testCreator",
            databasePath="/not-accessed",
        )
        self.session = SimpleNamespace(
            sessionID=4 if bound else None,
            _checkpointBinding={},
            _checkpointBaseline={},
            checkpointName="original",
            description="original description",
            mri=None,
            mne=None,
            dataMatrices={},
            actionHistory=[{"legacySessionID": 4}],
        )

        if bound:
            self.session._checkpointBinding = saveCoordinators._prepareCheckpointBinding(
                4, self.postgres,
                databaseID=self.databaseID,
                databaseName="originalProfile",
            )

        self.original = self.bookkeeping()
        self.originalHistory = deepcopy(self.session.actionHistory)

    def bookkeeping(self):
        return deepcopy({
            "sessionID": self.session.sessionID,
            "binding": self.session._checkpointBinding,
            "baseline": self.session._checkpointBaseline,
            "checkpointName": self.session.checkpointName,
            "description": self.session.description,
        })

    def requireOriginalBookkeeping(self):
        if self.bookkeeping() != self.original:
            raise AssertionError("Working checkpoint bookkeeping changed before connection cleanup.")

    def run(self):
        scenario = self

        class Coordinator(saveCoordinators):
            @classmethod
            def getSettings(cls, **kwargs):
                scenario.events.append("settingsResolved")
                return scenario.settings

            @classmethod
            def _getCheckpointHistoryBoundary(cls, session):
                return None

            @classmethod
            def _checkRawSessionLabels(cls, session, **kwargs):
                return True

            @classmethod
            def _requireStableDatabaseIdentity(cls, connection, postgres):
                scenario.requireOriginalBookkeeping()
                scenario.events.append("identityChecked")
                if scenario.failure == "compatibility":
                    raise RuntimeError("Simulated incompatible migration history.")
                if scenario.failure == "identity":
                    return "067ec061-c8cb-4b41-945c-315dc7af5fa4"
                return scenario.databaseID

            @classmethod
            def _acquireCheckpointPublicationLock(cls, connection, postgres, **kwargs):
                scenario.events.append("publicationLock")

            @classmethod
            def _getConfiguredStorageBackend(cls, connection, postgres, **kwargs):
                scenario.events.append("backendResolved")
                return 0

            @classmethod
            def getOrCreateUser(cls, connection, postgres, username):
                scenario.events.append("creatorResolved")
                return 0

            @classmethod
            def _saveRawSession(cls, *args, **kwargs):
                return cls.publish()

            @classmethod
            def _saveDerivedSession(cls, *args, **kwargs):
                return cls.publish()

            @classmethod
            def publish(cls):
                scenario.requireOriginalBookkeeping()
                scenario.events.append("publication")
                if scenario.failure == "publication":
                    raise ConnectionError("Simulated publication failure.")
                return 0

            @classmethod
            def _prepareCheckpointBaseline(cls, session, sessionID, postgres):
                scenario.requireOriginalBookkeeping()
                scenario.events.append("baselinePrepared")
                return {"testBaseline": sessionID}

            @classmethod
            def _handleCheckpointSaveFailure(cls, error, **kwargs):
                # Preserve unexpected failures; handle only the injected
                # operational failure category used by these tests.
                return isinstance(error, ConnectionError)

        namespace["pglPostgres"] = SimpleNamespace(
            validateTimeouts=lambda **kwargs: None,
            fromSettings=lambda **kwargs: scenario.postgres,
        )

        method = Coordinator.saveRawSession if self.raw else Coordinator.saveSession
        return method(self.session, settings=self.settings)

class SaveTransactionBoundaryTests(unittest.TestCase):
    def testSuccessAfterCleanup(self):
        for raw in (False, True):
            with self.subTest(raw=raw):
                scenario = SaveScenario(raw=raw)
                self.assertEqual(scenario.run(), 0)
                self.assertEqual(scenario.session.sessionID, 0)
                reference = scenario.session._checkpointBinding["checkpoint"]
                self.assertEqual(reference, {
                    "databaseID": scenario.databaseID,
                    "sessionID": 0,
                    "databaseName": "explicitProfile",
                })
                self.assertIn("commitConfirmed", scenario.events)
                self.assertIn("cleanupFinished", scenario.events)
                self.assertIsNot(scenario.session._checkpointPostgres, scenario.postgres)
                self.assertEqual(scenario.session.actionHistory, scenario.originalHistory)

    def testRawSaveDoesNotCreateHistoryBaseline(self):
        scenario = SaveScenario(raw=True, bound=False)
        self.assertEqual(scenario.run(), 0)
        self.assertEqual(scenario.session._checkpointBaseline, {})
        self.assertNotIn("baselinePrepared", scenario.events)
        self.assertEqual(scenario.session.actionHistory, scenario.originalHistory)

    def testRawSavePreservesExistingHistoryBaseline(self):
        scenario = SaveScenario(raw=True)
        scenario.session._checkpointBaseline = {
            "databaseIdentity": saveCoordinators._checkpointDatabaseIdentity(scenario.postgres),
            "testHistoricalBoundary": 7,
        }
        scenario.original = scenario.bookkeeping()
        self.assertEqual(scenario.run(), 0)
        self.assertEqual(scenario.session._checkpointBaseline, scenario.original["baseline"])

    def testDerivedSaveInstallsPreparedBaseline(self):
        scenario = SaveScenario()
        self.assertEqual(scenario.run(), 0)
        self.assertEqual(scenario.session._checkpointBaseline, {"testBaseline": 0})
        self.assertLess(scenario.events.index("baselinePrepared"), scenario.events.index("commitAttempted"))

    def testPublicationFailureLeavesBookkeepingUnchanged(self):
        for raw in (False, True):
            with self.subTest(raw=raw):
                scenario = SaveScenario(raw=raw, failure="publication")
                self.assertIsNone(scenario.run())
                scenario.requireOriginalBookkeeping()
                self.assertIn("rollback", scenario.events)
                self.assertNotIn("commitAttempted", scenario.events)

    def testUncertainCommitLeavesBookkeepingUnchanged(self):
        for raw in (False, True):
            with self.subTest(raw=raw):
                scenario = SaveScenario(raw=raw, failure="commit")
                self.assertIsNone(scenario.run())
                scenario.requireOriginalBookkeeping()
                self.assertIn("commitAttempted", scenario.events)
                self.assertNotIn("commitConfirmed", scenario.events)
                self.assertFalse(hasattr(scenario.session, "_checkpointPostgres"))

    def testCleanupFailureLeavesBookkeepingUnchanged(self):
        for raw in (False, True):
            with self.subTest(raw=raw):
                scenario = SaveScenario(raw=raw, failure="cleanup")
                self.assertIsNone(scenario.run())
                scenario.requireOriginalBookkeeping()
                self.assertIn("commitConfirmed", scenario.events)
                self.assertNotIn("cleanupFinished", scenario.events)
                self.assertFalse(hasattr(scenario.session, "_checkpointPostgres"))

    def testCancellationLeavesBookkeepingUnchanged(self):
        for raw in (False, True):
            with self.subTest(raw=raw):
                scenario = SaveScenario(raw=raw, failure="cancel")
                self.assertIsNone(scenario.run())
                scenario.requireOriginalBookkeeping()
                self.assertNotIn("transactionEntered", scenario.events)
                self.assertNotIn("publication", scenario.events)

    def testUUIDMismatchBlocksBeforePublicationWrites(self):
        for raw in (False, True):
            with self.subTest(raw=raw):
                scenario = SaveScenario(raw=raw, failure="identity")
                with self.assertRaisesRegex(ValueError, "UUID"):
                    scenario.run()
                scenario.requireOriginalBookkeeping()
                self.assertNotIn("backendResolved", scenario.events)
                self.assertNotIn("creatorResolved", scenario.events)
                self.assertNotIn("publication", scenario.events)

    def testMigrationMismatchBlocksBeforePublicationWrites(self):
        for raw in (False, True):
            with self.subTest(raw=raw):
                scenario = SaveScenario(raw=raw, failure="compatibility")
                with self.assertRaisesRegex(RuntimeError, "migration"):
                    scenario.run()
                scenario.requireOriginalBookkeeping()
                self.assertNotIn("backendResolved", scenario.events)
                self.assertNotIn("creatorResolved", scenario.events)
                self.assertNotIn("publication", scenario.events)

    def testEndpointMismatchBlocksBeforeConnection(self):
        for raw in (False, True):
            with self.subTest(raw=raw):
                scenario = SaveScenario(raw=raw)
                scenario.postgres.connectionHost = "different.example"
                with self.assertRaisesRegex(ValueError, "endpoint"):
                    scenario.run()
                scenario.requireOriginalBookkeeping()
                self.assertNotIn("connectionEntered", scenario.events)

    def testIdentityCheckPrecedesPublication(self):
        for raw in (False, True):
            with self.subTest(raw=raw):
                scenario = SaveScenario(raw=raw)
                scenario.run()
                self.assertLess(scenario.events.index("identityChecked"), scenario.events.index("backendResolved"))
                self.assertLess(scenario.events.index("identityChecked"), scenario.events.index("publication"))

suite = unittest.defaultTestLoader.loadTestsFromTestCase(SaveTransactionBoundaryTests)
testResult = unittest.TextTestRunner(verbosity=2).run(suite)
if not testResult.wasSuccessful(): raise AssertionError("Save transaction-boundary tests failed.")

In [ ]:
from pathlib import Path
import json
import os
import re
import shlex
import shutil
import subprocess
import tempfile
import uuid

import psycopg
from psycopg import sql

# Optional: set this to the directory containing postgres/initdb/pg_ctl.
# None searches existing PostgreSQL 17 installations and PATH.
postgresBinDirectory = None

if os.name != "posix": raise RuntimeError("This isolated Unix-socket setup requires macOS or Linux.")
if os.geteuid() == 0: raise RuntimeError("Run this as your normal user, not root.")
if globals().get("pglTestEnvironment") is not None:
    raise RuntimeError("A test environment is already recorded in this notebook. Stop it before creating another.")

def findTestPostgresBinaries(explicitDirectory=None):
    """Find an existing installation without installing or starting anything."""
    candidates = []

    if explicitDirectory is not None:
        candidates.append(Path(explicitDirectory).expanduser())
    else:
        brewExecutable = shutil.which("brew")
        if brewExecutable:
            result = subprocess.run(
                [brewExecutable, "--prefix", "postgresql@17"],
                capture_output=True, text=True, check=False,
            )
            if result.returncode == 0: candidates.append(Path(result.stdout.strip()) / "bin")

        candidates.extend([
            Path("/opt/homebrew/opt/postgresql@17/bin"),
            Path("/usr/local/opt/postgresql@17/bin"),
        ])
        for executableName in ("postgres", "initdb", "pg_ctl"):
            executable = shutil.which(executableName)
            if executable: candidates.append(Path(executable).resolve().parent)

    for candidate in candidates:
        directory = candidate.resolve()
        if all((directory / name).is_file() and os.access(directory / name, os.X_OK) for name in ("postgres", "initdb", "pg_ctl")):
            result = subprocess.run(
                [str(directory / "postgres"), "--version"],
                capture_output=True, text=True, check=True,
            )
            match = re.search(r"PostgreSQL\)\s+(\d+)", result.stdout)
            if match is None or int(match.group(1)) < 13:
                raise RuntimeError("Identity migration tests require PostgreSQL 13 or newer.")
            return directory, result.stdout.strip()

    raise FileNotFoundError(
        "Existing PostgreSQL binaries were not found. Set postgresBinDirectory "
        "to their bin directory. Nothing was installed or started."
    )

testBinDirectory, testServerVersion = findTestPostgresBinaries(postgresBinDirectory)

# Short socket paths avoid Unix socket path-length limits on macOS.
testRoot = Path(tempfile.mkdtemp(prefix="pgl-test-", dir="/tmp")).resolve()
testRoot.chmod(0o700)
testDataDirectory = testRoot / "data"
testSocketDirectory = testRoot / "socket"
testSocketDirectory.mkdir(mode=0o700)
testPayloadDirectory = testRoot / "payloads"
testPayloadDirectory.mkdir(mode=0o700)
testLogPath = testRoot / "postgres.log"

testEnvironment = {
    "kind": "pglDisposablePostgresTest",
    "version": 1,
    "environmentID": str(uuid.uuid4()),
    "root": str(testRoot),
    "dataDirectory": str(testDataDirectory),
    "socketDirectory": str(testSocketDirectory),
    "payloadDirectory": str(testPayloadDirectory),
    "logPath": str(testLogPath),
    "binDirectory": str(testBinDirectory),
    "serverVersion": testServerVersion,
    # This identifies the socket filename only. TCP listening is disabled.
    "port": 55432,
    "adminUser": "pgl_test_admin",
    "applicationUser": "pgl_test_app",
    "databaseNames": ["pgl_test_source", "pgl_test_destination"],
    "schema": "pgl",
}
testManifestPath = testRoot / "test-environment.json"
with testManifestPath.open("x", encoding="utf-8") as target:
    json.dump(testEnvironment, target, indent=2)
testManifestPath.chmod(0o600)

# Retain recovery information even if initialization or startup fails.
pglTestEnvironment = testEnvironment
pglTestManifestPath = testManifestPath

def runTestPostgresCommand(arguments):
    """Run only explicitly selected PostgreSQL binaries; retain diagnostics."""
    environment = dict(os.environ)
    environment["LC_ALL"] = "C"
    result = subprocess.run(
        [str(argument) for argument in arguments],
        env=environment, capture_output=True, text=True, check=False,
    )
    if result.returncode != 0:
        print(result.stdout)
        print(result.stderr)
        if testLogPath.is_file():
            print("\nServer log tail:")
            print(testLogPath.read_text(encoding="utf-8", errors="replace")[-12000:])
        raise RuntimeError(
            f"Test PostgreSQL command failed ({result.returncode}). "
            f"Files retained at {testRoot}"
        )
    return result

print(f"Using: {testServerVersion}")
print(f"Private test root: {testRoot}")
print(f"Recovery manifest: {testManifestPath}")

runTestPostgresCommand([
    testBinDirectory / "initdb",
    "-D", testDataDirectory,
    "--username", testEnvironment["adminUser"],
    "--encoding=UTF8",
    "--locale=C",
    "--auth-local=trust",
    "--auth-host=reject",
])

# Explicit command-line settings override cluster defaults.
# The private directory plus 0700 socket permissions restrict local access.
serverOptions = shlex.join([
    "-h", "",
    "-k", str(testSocketDirectory),
    "-p", str(testEnvironment["port"]),
    "-c", "unix_socket_permissions=0700",
])
runTestPostgresCommand([
    testBinDirectory / "pg_ctl",
    "-D", testDataDirectory,
    "-l", testLogPath,
    "-o", serverOptions,
    "-t", "30",
    "-w", "start",
])

def connectTestDatabase(databaseName="postgres", *, application=False):
    """Connect explicitly to this test socket, never default PGL/libpq routing."""
    environment = pglTestEnvironment
    if environment is None: raise RuntimeError("No active test environment is recorded.")
    if databaseName not in ["postgres", *environment["databaseNames"]]:
        raise ValueError("Only this environment's test databases may be selected.")

    connection = psycopg.connect(
        host=environment["socketDirectory"],
        port=environment["port"],
        dbname=databaseName,
        user=environment["applicationUser"] if application else environment["adminUser"],
        password="",
        sslmode="disable",
        connect_timeout=5,
        autocommit=True,
        application_name="pgl-disposable-tests",
        options="-c default_transaction_read_only=off",
    )

    try:
        actualDatabase, actualUser = connection.execute("SELECT current_database(), current_user").fetchone()
        expectedUser = environment["applicationUser"] if application else environment["adminUser"]
        if (actualDatabase, actualUser) != (databaseName, expectedUser):
            raise RuntimeError("Unexpected test connection identity.")

        # Administrative connections independently verify the exact cluster
        # before creating or modifying anything.
        if not application:
            actualDataDirectory = Path(connection.execute("SHOW data_directory").fetchone()[0]).resolve()
            if actualDataDirectory != Path(environment["dataDirectory"]):
                raise RuntimeError("Refusing access: connection reached another PostgreSQL cluster.")
            if connection.execute("SHOW listen_addresses").fetchone() != ("",):
                raise RuntimeError("Test server unexpectedly permits TCP listening.")

        return connection

    except BaseException:
        connection.close()
        raise

# CREATE DATABASE is deliberately outside a transaction.
# Every operation targets the verified, newly initialized private cluster.
with connectTestDatabase() as connection:
    applicationRole = sql.Identifier(testEnvironment["applicationUser"])
    adminRole = sql.Identifier(testEnvironment["adminUser"])

    connection.execute(sql.SQL(
        "CREATE ROLE {} LOGIN NOSUPERUSER NOCREATEDB NOCREATEROLE "
        "NOREPLICATION NOBYPASSRLS"
    ).format(applicationRole))

    for databaseName in testEnvironment["databaseNames"]:
        databaseIdentifier = sql.Identifier(databaseName)
        connection.execute(sql.SQL("CREATE DATABASE {} OWNER {}").format(databaseIdentifier, adminRole))
        connection.execute(sql.SQL("REVOKE ALL ON DATABASE {} FROM PUBLIC").format(databaseIdentifier))
        connection.execute(sql.SQL("GRANT CONNECT ON DATABASE {} TO {}").format(databaseIdentifier, applicationRole))

for databaseName in testEnvironment["databaseNames"]:
    with connectTestDatabase(databaseName) as connection:
        with connection.transaction():
            schemaIdentifier = sql.Identifier(testEnvironment["schema"])
            connection.execute(sql.SQL("REVOKE CREATE ON SCHEMA public FROM PUBLIC"))
            connection.execute(sql.SQL("CREATE SCHEMA {} AUTHORIZATION {}").format(schemaIdentifier, adminRole))
            connection.execute(sql.SQL("REVOKE ALL ON SCHEMA {} FROM PUBLIC").format(schemaIdentifier))
            connection.execute(sql.SQL("GRANT USAGE ON SCHEMA {} TO {}").format(schemaIdentifier, applicationRole))

    # Verify unprivileged access without creating scientific tables.
    with connectTestDatabase(databaseName, application=True) as connection:
        assert connection.execute("SELECT current_database(), current_user").fetchone() == (
            databaseName, testEnvironment["applicationUser"],
        )

    (testPayloadDirectory / databaseName).mkdir(mode=0o700)
    print(f"Ready: {databaseName} — administrator and application access verified")

print("\nTEST ENVIRONMENT READY")
print("TCP listening: disabled")
print("Scientific migrations: NOT applied")
print("Working PGL profiles/databases: untouched")
print(f"Manifest: {testManifestPath}")
print("Keep this environment running for the integration tests.")

In [ ]:
from pathlib import Path
import json
import os
import subprocess

# After a kernel restart, replace this with the printed manifest path.
manifestPath = Path("/private/tmp/pgl-test-4dl8lh7m/test-environment.json")

record = json.loads(manifestPath.read_text(encoding="utf-8"))
if record.get("kind") != "pglDisposablePostgresTest" or record.get("version") != 1:
    raise ValueError("Not a recognized disposable-test manifest.")

root = manifestPath.parent.resolve()
dataDirectory = Path(record["dataDirectory"]).resolve()
if root != Path(record["root"]).resolve() or dataDirectory != root / "data":
    raise ValueError("Test manifest paths are inconsistent.")
if not root.name.startswith("pgl-test-") or not (dataDirectory / "PG_VERSION").is_file():
    raise ValueError("Refusing to stop an unrecognized test cluster.")

pgCtl = Path(record["binDirectory"]) / "pg_ctl"
environment = {**os.environ, "LC_ALL": "C"}

status = subprocess.run(
    [str(pgCtl), "-D", str(dataDirectory), "status"],
    env=environment, capture_output=True, text=True, check=False,
)

if status.returncode == 0:
    subprocess.run(
        [str(pgCtl), "-D", str(dataDirectory), "-m", "fast", "-t", "30", "-w", "stop"],
        env=environment, check=True,
    )
elif status.returncode != 3:
    raise RuntimeError(f"Could not establish test server status:\n{status.stdout}\n{status.stderr}")

pglTestEnvironment = None
print(f"Test server stopped. All test files retained at: {root}")

In [ ]:
from pathlib import Path
import json
import os
import shlex
import subprocess

import psycopg

manifestPath = Path("/private/tmp/pgl-test-4dl8lh7m/test-environment.json")
record = json.loads(manifestPath.read_text(encoding="utf-8"))

if record.get("kind") != "pglDisposablePostgresTest" or record.get("version") != 1:
    raise ValueError("Not a recognized disposable-test manifest.")

root = manifestPath.parent.resolve()
dataDirectory = Path(record["dataDirectory"]).resolve()
socketDirectory = Path(record["socketDirectory"]).resolve()
binDirectory = Path(record["binDirectory"]).resolve()

# Refuse inconsistent paths before issuing any server command.
if root != Path(record["root"]).resolve() or dataDirectory != root / "data" or socketDirectory != root / "socket":
    raise ValueError("Test environment paths are inconsistent.")
if not root.name.startswith("pgl-test-") or not (dataDirectory / "PG_VERSION").is_file():
    raise ValueError("Not an initialized disposable test cluster.")
if not socketDirectory.is_dir(): raise FileNotFoundError(socketDirectory)
if root.stat().st_uid != os.getuid() or root.stat().st_mode & 0o077:
    raise PermissionError("Test root must be owned by you and inaccessible to other users.")

environment = {**os.environ, "LC_ALL": "C"}
pgCtl = binDirectory / "pg_ctl"

status = subprocess.run(
    [str(pgCtl), "-D", str(dataDirectory), "status"],
    env=environment, capture_output=True, text=True, check=False,
)

if status.returncode == 3:
    # Preserve private socket access and explicitly disable TCP listening.
    serverOptions = shlex.join([
        "-h", "",
        "-k", str(socketDirectory),
        "-p", str(record["port"]),
        "-c", "unix_socket_permissions=0700",
    ])
    subprocess.run(
        [
            str(pgCtl), "-D", str(dataDirectory),
            "-l", record["logPath"],
            "-o", serverOptions,
            "-t", "30", "-w", "start",
        ],
        env=environment, check=True,
    )
elif status.returncode != 0:
    raise RuntimeError(f"Could not establish test server status:\n{status.stdout}\n{status.stderr}")

# Verify the actual server before restoring notebook bookkeeping.
for databaseName in record["databaseNames"]:
    with psycopg.connect(
        host=str(socketDirectory),
        port=record["port"],
        dbname=databaseName,
        user=record["adminUser"],
        password="",
        sslmode="disable",
        connect_timeout=5,
        autocommit=True,
        application_name="pgl-disposable-tests",
    ) as connection:
        actualDirectory = Path(connection.execute("SHOW data_directory").fetchone()[0]).resolve()
        if actualDirectory != dataDirectory:
            raise RuntimeError("Connection reached an unexpected PostgreSQL cluster.")
        if connection.execute("SHOW listen_addresses").fetchone() != ("",):
            raise RuntimeError("Test server unexpectedly permits TCP listening.")
        if connection.execute("SELECT current_database(), current_user").fetchone() != (databaseName, record["adminUser"]):
            raise RuntimeError("Unexpected test connection identity.")

    print(f"Verified: {databaseName}")

pglTestEnvironment = record
pglTestManifestPath = manifestPath

print("\nTest environment running again; TCP disabled.")
print("No migrations applied or databases recreated.")
print("Leave it running—do not run the shutdown cell until testing is finished.")

In [27]:
from pathlib import Path
import subprocess
import sys
import tempfile

repoPath = Path("~/proj/pgl").expanduser().resolve()
manifestPath = Path("/private/tmp/pgl-test-4dl8lh7m/test-environment.json")
fixturePath = manifestPath.parent / "session-roundtrip-va7c1poe" / "result.json"

previousTestPath = Path(
    "/var/folders/n2/4bqdpl1s6wd9x4l9ll4p03_m0000gq/T/"
    "pgl-export-manifest-tests-4dfmo5ys/exportManifestTests.py"
)
if not previousTestPath.is_file(): raise FileNotFoundError(previousTestPath)

testCode = previousTestPath.read_text(encoding="utf-8")
anchor = "suite = unittest.defaultTestLoader.loadTestsFromTestCase(ExportManifestTests)"
if testCode.count(anchor) != 1:
    raise ValueError("Retained manifest test script differs from expectations.")
if "class ExportFileInventoryTests" in testCode:
    raise ValueError("Retained script already contains inventory tests.")

additionalTests = r'''
class ExportFileInventoryTests(ExportManifestTests):
    # Inherit the same real read-only setup, but select only the new tests
    # below when constructing the suite.

    def inventory(self, checkpoint=None, configuration=None):
        return pglDatabase._readExportFileInventory(
            self.connection,
            postgres if configuration is None else configuration,
            self.ancestry[-1] if checkpoint is None else checkpoint,
        )

    def testFileInventoryIncludesManifestAndAttachments(self):
        result = self.inventory()
        records = {record["name"]: record for record in result["files"]}
        self.assertIn("manifest.json", records)
        self.assertEqual(records["manifest.json"]["role"], "manifest")
        self.assertEqual(records["manifest.json"]["sha256"], self.ancestry[-1]["manifestHash"])

        manifest = result["manifest"]
        expectedNames = {record["name"] for record in manifest["files"]} | {"manifest.json"}
        self.assertEqual(set(records), expectedNames)
        for attachment in manifest["derivedFiles"]:
            self.assertIn(attachment["path"], records)
        self.assertTrue(all(record["locations"] for record in records.values()))
        self.assertFalse(result["matrixInventoryComplete"])

    def testByteCountsAreExplicitlyFileOnly(self):
        result = self.inventory()
        self.assertEqual(result["logicalBytes"], sum(record["size_bytes"] for record in result["files"]))
        unique = {record["fileID"]: record["size_bytes"] for record in result["files"]}
        self.assertEqual(result["uniqueContentBytes"], sum(unique.values()))
        self.assertLessEqual(result["uniqueContentBytes"], result["logicalBytes"])

    def testCatalogReadDoesNotProbeScientificLocations(self):
        from pgl.pglDatabase import pglDatabaseAccess

        # Manifest verification necessarily opens its source location.
        # After that real validation finishes, prohibit all location resolution.
        original = pglDatabase._readExportCheckpointManifest
        blocker = None

        def validateThenBlock(*args, **kwargs):
            nonlocal blocker
            manifest = original(*args, **kwargs)
            blocker = patch.object(
                pglDatabaseAccess, "resolve",
                side_effect=AssertionError("Inventory probed a payload location."),
            )
            blocker.start()
            return manifest

        try:
            with patch.object(pglDatabase, "_readExportCheckpointManifest", side_effect=validateThenBlock):
                self.assertTrue(self.inventory()["files"])
        finally:
            if blocker is not None: blocker.stop()

    def testReturnedLocationListsAreIndependent(self):
        first = self.inventory()
        second = self.inventory()
        first["files"][0]["locations"].clear()
        self.assertTrue(second["files"][0]["locations"])

    def shadowConfiguration(self, *, missingLocations=False, corruptRole=False, deletedContent=False):
        """Present a test-only catalog projection without editing saved rows.

        PostgreSQL permits temporary-object DDL in a read-only transaction.
        All production identifiers stay schema-qualified; this configuration
        explicitly selects pg_temp only inside these tests.

        The real manifest bytes are still read/hashed by the existing reader.
        """
        from psycopg import sql

        checkpointID = self.ancestry[-1]["sessionID"]
        if missingLocations:
            # Keep manifest locations readable so failure occurs during
            # complete payload inventory, not manifest acquisition.
            self.connection.execute(sql.SQL("""
                CREATE TEMP VIEW storage_locations AS
                SELECT location.*
                FROM pgl.storage_locations AS location
                WHERE NOT EXISTS (
                    SELECT 1 FROM pgl.session_files AS link
                    WHERE link.session_id = {}
                      AND link.file_id = location.file_id
                      AND link.name = 'history.json'
                )
            """).format(sql.Literal(checkpointID)))
        else:
            self.connection.execute("CREATE TEMP VIEW storage_locations AS SELECT * FROM pgl.storage_locations")

        # Enumerate columns so an altered role/deletion marker replaces its
        # original value, rather than creating ambiguous duplicate columns.
        self.connection.execute(sql.SQL("""
            CREATE TEMP VIEW session_files AS
            SELECT session_id, name, file_id,
                   CASE WHEN {} AND session_id = {} AND name = 'history.json'
                        THEN 'unexpectedRole' ELSE role END AS role,
                   subject_id, deleted_at
            FROM pgl.session_files
        """).format(sql.Literal(corruptRole), sql.Literal(checkpointID)))

        self.connection.execute(sql.SQL("""
            CREATE TEMP VIEW files AS
            SELECT file_id, sha256, size_bytes, format, kind,
                   created_at, unreferenced_since,
                   CASE WHEN {} AND file_id IN (
                       SELECT file_id FROM pgl.session_files
                       WHERE session_id = {} AND name = 'history.json'
                   ) THEN statement_timestamp() ELSE deleted_at END AS deleted_at
            FROM pgl.files
        """).format(sql.Literal(deletedContent), sql.Literal(checkpointID)))

        for name in ("session", "schema_identity", "storage_backends", "session_products", "matrix_revisions"):
            self.connection.execute(
                sql.SQL("CREATE TEMP VIEW {} AS SELECT * FROM {}").format(
                    sql.Identifier(name), sql.Identifier("pgl", name),
                )
            )

        configuration = postgres.withTimeouts()
        # This test-only namespace routes reads to explicitly constructed views.
        configuration.databaseSchema = "pg_temp"
        return configuration

    def testMissingPayloadLocationsBlocked(self):
        # Database-level TEMP privilege is intentionally not granted to the
        # application role. Restore the verified test administrator solely
        # to construct temporary projections.
        self.connection.execute("RESET ROLE")
        configuration = self.shadowConfiguration(missingLocations=True)
        with self.assertRaisesRegex(ValueError, "without eligible registered locations"):
            self.inventory(configuration=configuration)

    def testUnexpectedFileRoleBlocked(self):
        self.connection.execute("RESET ROLE")
        configuration = self.shadowConfiguration(corruptRole=True)
        with self.assertRaisesRegex(ValueError, "exactly match"):
            self.inventory(configuration=configuration)

    def testDeletedContentBlocked(self):
        self.connection.execute("RESET ROLE")
        configuration = self.shadowConfiguration(deletedContent=True)
        with self.assertRaisesRegex(ValueError, "deleted link or content"):
            self.inventory(configuration=configuration)

newTestNames = [
    "testFileInventoryIncludesManifestAndAttachments",
    "testByteCountsAreExplicitlyFileOnly",
    "testCatalogReadDoesNotProbeScientificLocations",
    "testReturnedLocationListsAreIndependent",
    "testMissingPayloadLocationsBlocked",
    "testUnexpectedFileRoleBlocked",
    "testDeletedContentBlocked",
]
suite = unittest.TestSuite(
    ExportFileInventoryTests(name) for name in newTestNames
)
'''

testCode = testCode.replace(anchor, additionalTests, 1)

testDirectory = Path(tempfile.mkdtemp(prefix="pgl-export-file-inventory-tests-"))
testPath = testDirectory / "exportFileInventoryTests.py"
testPath.write_text(testCode, encoding="utf-8")
stdoutPath = testDirectory / "stdout.log"
stderrPath = testDirectory / "stderr.log"

with stdoutPath.open("xb") as stdout, stderrPath.open("xb") as stderr:
    completed = subprocess.run(
        [sys.executable, str(testPath), str(repoPath), str(manifestPath), str(fixturePath)],
        cwd=repoPath,
        stdout=stdout,
        stderr=stderr,
        check=False,
    )

for path in (stdoutPath, stderrPath):
    print(f"\n=== {path.name} ===")
    print(path.read_text(encoding="utf-8", errors="replace") or "(empty)")

print(f"Test source and diagnostics: {testDirectory}")
if completed.returncode != 0:
    raise AssertionError("Export file-inventory tests failed; send the diagnostics above.")

print("File inventory tests passed. No persistent database or payload changes.")


=== stdout.log ===
(empty)

=== stderr.log ===
testFileInventoryIncludesManifestAndAttachments (__main__.ExportFileInventoryTests.testFileInventoryIncludesManifestAndAttachments) ... ok
testByteCountsAreExplicitlyFileOnly (__main__.ExportFileInventoryTests.testByteCountsAreExplicitlyFileOnly) ... ok
testCatalogReadDoesNotProbeScientificLocations (__main__.ExportFileInventoryTests.testCatalogReadDoesNotProbeScientificLocations) ... ok
testReturnedLocationListsAreIndependent (__main__.ExportFileInventoryTests.testReturnedLocationListsAreIndependent) ... ok
testMissingPayloadLocationsBlocked (__main__.ExportFileInventoryTests.testMissingPayloadLocationsBlocked) ... ERROR
testUnexpectedFileRoleBlocked (__main__.ExportFileInventoryTests.testUnexpectedFileRoleBlocked) ... ERROR
testDeletedContentBlocked (__main__.ExportFileInventoryTests.testDeletedContentBlocked) ... ERROR

ERROR: testMissingPayloadLocationsBlocked (__main__.ExportFileInventoryTests.testMissingPayloadLocationsBlocked)
----

AssertionError: Export file-inventory tests failed; send the diagnostics above.

In [31]:
from pathlib import Path
import ast
import json
import subprocess
import sys
import tempfile

repoPath = Path("~/proj/pgl").expanduser().resolve()
manifestPath = Path("/private/tmp/pgl-test-4dl8lh7m/test-environment.json")
previousTestPath = manifestPath.parent / "session-roundtrip-va7c1poe" / "behaviorRoundTripTests.py"

environment = json.loads(manifestPath.read_text(encoding="utf-8"))
testRoot = manifestPath.parent.resolve()

if environment.get("kind") != "pglDisposablePostgresTest" or environment.get("version") != 1:
    raise ValueError("Not a recognized disposable-test environment.")
if Path(environment["root"]).resolve() != testRoot:
    raise ValueError("Test root mismatch.")
if not previousTestPath.is_file(): raise FileNotFoundError(previousTestPath)

testCode = previousTestPath.read_text(encoding="utf-8")

def replaceMatrixTestSource(old, new):
    """Verify the successful fixture before adapting it; never edit its file."""
    global testCode
    if testCode.count(old) != 1:
        raise ValueError(f"Expected exactly one test-source match:\n{old[:250]}")
    testCode = testCode.replace(old, new, 1)

replaceMatrixTestSource(
    "import numpy as np\n",
    "import numpy as np\n"
    "import xarray as xr\n",
)
replaceMatrixTestSource(
    "from pgl.pglSession import pglSession\n",
    "from pgl.pglSession import pglSession\n"
    "from pgl.pglData import pglDataMatrix\n",
)

# Attach a real working matrix before the first derived save. Raw publication
# remains acquisition-only, and the existing action history is unchanged.
replaceMatrixTestSource(
    "            beforeSave = historySnapshot(session)\n",
    '''            expectedMatrix = xr.Dataset(
                data_vars={
                    "signal": (("sample",), np.array([0.0, 1.25, -2.5, 4.0], dtype=np.float64)),
                    "valid": (("sample",), np.array([False, True, True, False], dtype=bool)),
                },
                coords={"sample": np.arange(4, dtype=np.int64)},
                attrs={"fixture": "syntheticExportMatrix"},
            )
            matrix = pglDataMatrix(expectedMatrix.copy(deep=True), name="syntheticMatrix")
            matrix.provenance = {
                "operation": "syntheticIntegrationFixture",
                "description": "Four synthetic samples; no acquired data.",
            }
            session.dataMatrices = {"syntheticMatrix": matrix}

            beforeSave = historySnapshot(session)
''',
)

# Verify actual stored values, not only the descriptor.
replaceMatrixTestSource(
    '            self.assertEqual(reference(loaded)["sessionID"], derivedID)\n',
    '''            self.assertEqual(reference(loaded)["sessionID"], derivedID)
            self.assertEqual(set(loaded.dataMatrices), {"syntheticMatrix"})

            restoredMatrix = loaded.dataMatrices["syntheticMatrix"]
            xr.testing.assert_identical(
                restoredMatrix.requireData().compute(),
                expectedMatrix,
            )
            self.assertEqual(
                restoredMatrix.provenance["operation"],
                "syntheticIntegrationFixture",
            )

            # Check real registered matrix metadata through the new inspector.
            # This query must not open the matrix bundle again.
            with postgres.connect() as connection:
                with connection.transaction():
                    connection.execute("SET TRANSACTION ISOLATION LEVEL REPEATABLE READ, READ ONLY")
                    pglDatabase._requireStableDatabaseIdentity(connection, postgres)
                    ancestry = pglDatabase._readExportAncestry(connection, postgres, derivedID)

                    with patch.object(
                        pglDatabase, "_openRegisteredMatrix",
                        side_effect=AssertionError("Catalog inventory must not open a matrix."),
                    ):
                        rawMatrixInventory = pglDatabase._readExportMatrixInventory(
                            connection, postgres, ancestry[0],
                        )
                        matrixInventory = pglDatabase._readExportMatrixInventory(
                            connection, postgres, ancestry[-1],
                        )

            self.assertEqual(rawMatrixInventory, {
                "attachments": [],
                "revisions": [],
                "bundleInventoryComplete": True,
                "bundleBytes": 0,
            })
            self.assertEqual(matrixInventory["attachments"], [{
                "catalogName": "syntheticMatrix",
                "owner": "session",
                "localName": "syntheticMatrix",
                "matrixRevisionID": restoredMatrix.matrixRevisionID,
            }])
            self.assertEqual(len(matrixInventory["revisions"]), 1)
            revision = matrixInventory["revisions"][0]
            self.assertEqual(revision["matrixRevisionID"], restoredMatrix.matrixRevisionID)
            self.assertEqual(revision["contractVersion"], "pgl.matrix.zarr.v1")
            self.assertEqual(revision["registeredRoot"], str(payloadRoot))
            self.assertFalse(matrixInventory["bundleInventoryComplete"])
            self.assertIsNone(matrixInventory["bundleBytes"])

            # Independently compare the returned catalog descriptor with the
            # actual stored descriptor. Scientific chunk verification remains
            # a separate implementation task.
            bundlePath = payloadRoot / revision["objectKey"]
            storedDescriptor = json.loads(
                (bundlePath / "matrix.json").read_text(encoding="utf-8")
            )
            self.assertEqual(
                pglDatabase.encodeRawManifest(revision["descriptor"]),
                pglDatabase.encodeRawManifest(storedDescriptor),
            )
            self.assertTrue((bundlePath / "data.zarr").is_dir())
''',
)

replaceMatrixTestSource(
    '            self.assertEqual(reloaded.runs[0].data.endTime, 3.0)\n',
    '''            self.assertEqual(reloaded.runs[0].data.endTime, 3.0)
            self.assertEqual(set(reloaded.dataMatrices), {"syntheticMatrix"})
            xr.testing.assert_identical(
                reloaded.dataMatrices["syntheticMatrix"].requireData().compute(),
                expectedMatrix,
            )

            # Ordinary save currently publishes a new matrix revision.
            # Do not mistake unchanged working values for implemented reuse.
            self.assertNotEqual(
                reloaded.dataMatrices["syntheticMatrix"].matrixRevisionID,
                restoredMatrix.matrixRevisionID,
            )
''',
)

replaceMatrixTestSource(
    '                "runDirectory": str(runPath),\n',
    '''                "runDirectory": str(runPath),
                "matrixName": "syntheticMatrix",
                "firstMatrixRevisionID": restoredMatrix.matrixRevisionID,
                "secondMatrixRevisionID": reloaded.dataMatrices["syntheticMatrix"].matrixRevisionID,
                "firstMatrixObjectKey": revision["objectKey"],
                "matrixCatalogInventoryVerified": True,
                "completeBundleHashInventoryVerified": False,
''',
)

# Validate before creating another test invocation.
ast.parse(testCode)

runPath = Path(tempfile.mkdtemp(prefix="session-roundtrip-", dir=testRoot))
testPath = runPath / "matrixRoundTripTests.py"
stdoutPath = runPath / "subprocess.stdout.log"
stderrPath = runPath / "subprocess.stderr.log"
testPath.write_text(testCode, encoding="utf-8")

print(f"Retained test source, payloads, and diagnostics:\n{runPath}", flush=True)
print("Running real matrix publication/reload and catalog inventory...", flush=True)

with stdoutPath.open("xb") as stdout, stderrPath.open("xb") as stderr:
    completed = subprocess.run(
        [sys.executable, str(testPath), str(repoPath), str(manifestPath), str(runPath)],
        cwd=repoPath,
        stdout=stdout,
        stderr=stderr,
        check=False,
    )

def printMatrixTestLog(path, limit=24000):
    print(f"\n=== {path.name} ===")
    with path.open("rb") as source:
        source.seek(0, 2)
        size = source.tell()
        source.seek(max(0, size - limit))
        content = source.read()
    if size > limit:
        print(f"[Final {limit:,} bytes; complete log retained at {path}]")
    print(content.decode("utf-8", errors="replace") or "(empty)")

printMatrixTestLog(stdoutPath)
printMatrixTestLog(stderrPath)

if completed.returncode != 0:
    raise AssertionError(
        f"Matrix integration test failed. Send the stderr above. "
        f"Files retained at {runPath}; checkpoints may already have committed."
    )

print("\nReal matrix publication/reload and catalog inventory passed.")
print(f"Result record: {runPath / 'result.json'}")

Retained test source, payloads, and diagnostics:
/private/tmp/pgl-test-4dl8lh7m/session-roundtrip-wobnd0pj
Running real matrix publication/reload and catalog inventory...

=== subprocess.stdout.log ===
(pglDatabase:saveRawSession) Raw checkpoint 3 is complete. Database configuration: /private/tmp/pgl-test-4dl8lh7m/session-roundtrip-wobnd0pj (database=pgl_test_source, schema=pgl; 0.1 s total).
(pglDatabase:loadSession) Loading sessionID=3 from isolatedRoundTripProfile database: /private/tmp/pgl-test-4dl8lh7m/session-roundtrip-wobnd0pj,
username=roundtrip_test_creator
================================== Session 3 ===================================
  Acquired         : January 1, 2026 at 12:00 PM UTC
  Experiment       : pglIdentityIntegration
  Subject          : synthetic0001
  Runs             : 1
  MEG/EEG data     : None
  MRI data         : None
  Checkpoint       : Not set
  Checkpoint saved : November 21, 2026 at 6:27 PM PST
  Saved by         : roundtrip_test_creator
Action conso

AssertionError: Matrix integration test failed. Send the stderr above. Files retained at /private/tmp/pgl-test-4dl8lh7m/session-roundtrip-wobnd0pj; checkpoints may already have committed.

In [ ]:
from pathlib import Path
import json
import subprocess
import sys
import tempfile

allowCommitDerivedSourceFixture = True

repoPath = Path("/Users/justin/proj/pgl").resolve(strict=True)
environmentPath = Path(
    "/private/tmp/pgl-test-4dl8lh7m/test-environment.json"
).resolve(strict=True)
rawFixturePath = Path(
    "/private/tmp/pgl-test-4dl8lh7m/raw-registrar-success-epgvgm_0/"
    "source-fixture.json"
).resolve(strict=True)

if allowCommitDerivedSourceFixture is not True:
    raise RuntimeError(
        "Not executed. Set allowCommitDerivedSourceFixture = True to publish "
        "one derived synthetic source checkpoint beneath source checkpoint 8. "
        "No destination database will be opened; fixtures will be retained."
    )

testPath = Path(tempfile.mkdtemp(
    prefix="derived-registrar-source-", dir=environmentPath.parent,
)).resolve(strict=True)
scriptPath = testPath / "derivedSourceFixtureTests.py"

testSource = r'''
from contextlib import contextmanager, ExitStack
from pathlib import Path
from unittest.mock import patch
import hashlib
import inspect
import json
import os
import sys

repoPath, environmentPath, rawFixturePath, testPath = map(Path, sys.argv[1:])
sys.path.insert(0, str(repoPath))

import numpy as np
import xarray as xr
import psycopg
from psycopg import sql
from traitlets import Unicode

from pgl.pglDatabase import pglDatabase, pglDatabaseSettings
from pgl.pglPostgres import pglPostgres
from pgl.pglSession import pglSession
from pgl.pglPipeline import pglAction, pglActionStatus
from pgl.pglData import pglDataMatrix

testRoot = Path("/private/tmp/pgl-test-4dl8lh7m").resolve(strict=True)
databaseName = "pgl_test_source"
databaseID = "787cd1ab-63d5-439b-b880-f96d0fbfb5cb"
rawSessionID = 8
expectedRawHash = "39e6e9c76d483c6da068996fe9e425fce098e01c66db1faea5c9523480f9e8e8"
passed = []
loadedSessions = []
saveAttempted = False
saveReturnedID = None
publicSQLWritesAllowed = False

def require(condition, message):
    if not condition: raise AssertionError(message)

def writeJSON(name, document):
    (testPath / name).write_text(
        json.dumps(document, indent=2, allow_nan=False), encoding="utf-8",
    )

def recordPass(label):
    passed.append(label)
    writeJSON("passes.json", passed)
    print(f"PASS: {label}", flush=True)

def requireLocalSafe(path):
    for candidate in (path, *path.parents):
        require(not candidate.is_symlink(), f"Symlinked test path: {candidate}")

def snapshot(root):
    requireLocalSafe(root)
    require(root.is_dir(), f"Missing fixture root: {root}")
    files, directories = {}, []
    for path in sorted(root.rglob("*")):
        requireLocalSafe(path)
        name = path.relative_to(root).as_posix()
        if path.is_dir():
            directories.append(name)
            continue
        require(path.is_file(), f"Unsupported entry: {path}")
        digest = hashlib.sha256()
        size = 0
        with path.open("rb") as handle:
            while True:
                block = handle.read(1024 * 1024)
                if not block: break
                digest.update(block)
                size += len(block)
        files[name] = {"sha256": digest.hexdigest(), "size_bytes": size}
    return {"files": files, "directories": directories}

require(
    Path(inspect.getfile(pglDatabase)).resolve() == repoPath / "pgl/pglDatabase.py",
    "Imported a different PGL checkout.",
)
require(
    environmentPath == testRoot / "test-environment.json"
    and rawFixturePath == testRoot / "raw-registrar-success-epgvgm_0/source-fixture.json",
    "Unexpected test manifest paths.",
)
require(
    testPath.parent == testRoot and testPath.name.startswith("derived-registrar-source-"),
    "Unexpected workspace.",
)
rootStat = testRoot.stat()
require(
    rootStat.st_uid == os.getuid() and not rootStat.st_mode & 0o077,
    "Expected private owner-only disposable root.",
)

environmentBytes = environmentPath.read_bytes()
rawFixtureBytes = rawFixturePath.read_bytes()
environment = json.loads(environmentBytes)
rawFixture = json.loads(rawFixtureBytes)

for key, value in {
    "kind": "pglDisposablePostgresTest",
    "version": 1,
    "root": str(testRoot),
    "dataDirectory": str(testRoot / "data"),
    "socketDirectory": str(testRoot / "socket"),
    "port": 55432,
    "databaseNames": ["pgl_test_source", "pgl_test_destination"],
    "schema": "pgl",
    "adminUser": "pgl_test_admin",
    "applicationUser": "pgl_test_app",
}.items():
    require(environment.get(key) == value, f"Environment mismatch: {key}")
require(type(environment["port"]) is int, "Invalid port representation.")
require(type(environment["version"]) is int, "Invalid environment version.")

for key, value in {
    "kind": "pglDisposableRawRegistrarSourceFixture",
    "version": 1,
    "environmentManifest": str(environmentPath),
    "databaseName": databaseName,
    "databaseID": databaseID,
    "sessionID": rawSessionID,
    "manifestHash": expectedRawHash,
    "backendRoot": str(rawFixturePath.parent / "sourcePayloads"),
    "objectPrefix": "raw",
    "commitConfirmed": True,
}.items():
    require(rawFixture.get(key) == value, f"Raw fixture mismatch: {key}")

rawPayloadRoot = Path(rawFixture["backendRoot"])
rawBefore = snapshot(rawPayloadRoot)
writeJSON("rawPayloadsBefore.json", rawBefore)

payloadRoot = testPath / "payloads"
cacheRoot = testPath / "cache"
payloadRoot.mkdir(mode=0o700)
cacheRoot.mkdir(mode=0o700)

testSettings = pglDatabaseSettings(
    name="isolatedDerivedRegistrarSource",
    databasePath=str(testPath),
    cacheDir=str(cacheRoot),
    username="derived_registrar_fixture_creator",
)
postgres = pglPostgres(
    databaseName=databaseName,
    databaseSchema="pgl",
    databaseUser=environment["applicationUser"],
    adminUser=environment["adminUser"],
    postgresPort=environment["port"],
    installDirectory=str(testPath),
    storageLocations=[str(payloadRoot)],
    databaseSSLMode="disable",
)
expectedConfiguration = postgres.toJSONdict()

def openSocketConnection(user, *, allowWrites=False):
    require(
        user in (environment["adminUser"], environment["applicationUser"]),
        "Unexpected SQL role.",
    )
    require(
        not allowWrites or user == environment["applicationUser"],
        "Administrator writes are prohibited.",
    )
    return psycopg.connect(
        host=environment["socketDirectory"],
        port=environment["port"],
        dbname=databaseName,
        user=user,
        password="",
        sslmode="disable",
        connect_timeout=5,
        autocommit=True,
        application_name="pgl-derived-registrar-source-fixture",
        options=(
            f"-c default_transaction_read_only={'off' if allowWrites else 'on'} "
            "-c statement_timeout=60000 -c lock_timeout=5000"
        ),
    )

def checkIdentity(connection, user):
    require(
        connection.execute("SELECT current_database(), current_user").fetchone()
        == (databaseName, user),
        "Unexpected SQL connection identity.",
    )

@contextmanager
def testConnect(self, *, user=None, **kwargs):
    """Route public PGL operations only to this explicit source configuration."""
    require(self.toJSONdict() == expectedConfiguration, "Unexpected PGL configuration.")
    require(
        user is None or user == environment["applicationUser"],
        "Public operations must use the application role.",
    )
    with openSocketConnection(
        environment["applicationUser"], allowWrites=publicSQLWritesAllowed,
    ) as connection:
        checkIdentity(connection, environment["applicationUser"])
        connection.execute(
            sql.SQL("SET search_path TO {}, pg_catalog").format(sql.Identifier("pgl"))
        )
        yield connection

def explicitSettingsOnly(cls, settingsName=None, settings=None, returnSettingsList=False):
    require(
        settings is testSettings and settingsName is None and not returnSettingsList,
        "Unexpected default/profile lookup.",
    )
    return testSettings

def explicitConfigurationOnly(cls, settings=None, settingsName=None, **kwargs):
    require(
        settings is testSettings and settingsName is None and not kwargs,
        "Unexpected PostgreSQL configuration lookup.",
    )
    return postgres.withTimeouts()

class DerivedRegistrarFixtureAction(pglAction):
    """Advance one synthetic behavioral value through the real action runner."""
    cacheDirectory = Unicode("").tag(serialize=False)

    def _run(self, session):
        session.runs[0].data.endTime += 1.0
        return session

def historySnapshot(session):
    return [entry.toJSON(strict=True) for entry in session.actionHistory]

def readSessionRow(connection, sessionID):
    row = connection.execute(
        sql.SQL("SELECT to_jsonb(item) FROM {} AS item WHERE session_id = %s").format(
            sql.Identifier("pgl", "session")
        ), (sessionID,),
    ).fetchone()
    require(row is not None, "Expected session catalog row is absent.")
    return row[0]

try:
    with openSocketConnection(environment["adminUser"]) as connection:
        with connection.transaction():
            connection.execute(
                "SET TRANSACTION ISOLATION LEVEL REPEATABLE READ, READ ONLY"
            )
            checkIdentity(connection, environment["adminUser"])
            require(
                Path(connection.execute("SHOW data_directory").fetchone()[0]).resolve(strict=True)
                == Path(environment["dataDirectory"]).resolve(strict=True),
                "Connected to another PostgreSQL cluster.",
            )
            require(
                connection.execute("SHOW listen_addresses").fetchone() == ("",),
                "Disposable cluster unexpectedly permits TCP listening.",
            )
            require(
                pglDatabase._requireStableDatabaseIdentity(connection, postgres) == databaseID,
                "Stable source UUID mismatch.",
            )
    recordPass("explicit source cluster, checkout, migrations, and UUID verified")

    with openSocketConnection(environment["applicationUser"]) as connection:
        with connection.transaction():
            connection.execute(
                "SET TRANSACTION ISOLATION LEVEL REPEATABLE READ, READ ONLY"
            )
            require(
                pglDatabase._requireStableDatabaseIdentity(connection, postgres) == databaseID,
                "Application source UUID mismatch.",
            )
            rawCatalogBefore = readSessionRow(connection, rawSessionID)
            require(
                rawCatalogBefore["manifest_hash"] == expectedRawHash
                and rawCatalogBefore["status"] == "complete"
                and rawCatalogBefore["deleted_at"] is None
                and rawCatalogBefore["parent_session_id"] is None,
                "Raw source fixture is not the expected active completed root.",
            )
    writeJSON("rawCatalogBefore.json", rawCatalogBefore)

    with ExitStack() as guards:
        guards.enter_context(patch.object(
            pglDatabase, "getSettings", new=classmethod(explicitSettingsOnly),
        ))
        guards.enter_context(patch.object(
            pglPostgres, "fromSettings", new=classmethod(explicitConfigurationOnly),
        ))
        guards.enter_context(patch.object(pglPostgres, "connect", new=testConnect))
        for owner, name in (
            (pglPostgres, "_localConnect"),
            (pglPostgres, "start"),
            (pglPostgres, "createDatabase"),
            (pglDatabase, "initDatabase"),
        ):
            guards.enter_context(patch.object(
                owner, name,
                side_effect=AssertionError(f"Forbidden administrative operation: {name}"),
            ))

        session = pglSession.loadSession(
            rawSessionID, settings=testSettings, cacheDirectory=str(cacheRoot),
        )
        require(session is not None, "Raw fixture load failed.")
        loadedSessions.append(session)
        require(len(session.runs) == 1, "Expected one synthetic behavioral run.")
        require(session.actionHistory == [], "Raw fixture unexpectedly contains action history.")
        sourceReference = pglDatabase._getBoundCheckpointReference(session)
        require(sourceReference == {
            "databaseID": databaseID,
            "sessionID": rawSessionID,
            "databaseName": testSettings.name,
        }, "Loaded raw binding differs.")

        originalEndTime = session.runs[0].data.endTime
        acquisitionUUID = session.runs[0].data.uuid
        action = DerivedRegistrarFixtureAction()
        action.configure(cacheDirectory=str(cacheRoot))
        require(action.run(session=session) is session, "Action returned another object.")
        require(action.status == pglActionStatus.SUCCESS, "Synthetic action failed.")
        require(session.runs[0].data.endTime == originalEndTime + 1.0,
                "Synthetic action did not change the intended value.")
        require(len(session.actionHistory) == 1, "Expected one recorded action.")
        entry = session.actionHistory[0]
        require(
            entry.actionInput["inputs"][0]["checkpoint"] == sourceReference,
            "Action input does not identify the real raw source checkpoint.",
        )
        recordPass("real action records qualified input checkpoint 8 and changes synthetic data")

        expectedMatrix = xr.Dataset(
            data_vars={
                "signal": (("sample",), np.array([0.0, 1.25, -2.5, 4.0], dtype=np.float64)),
                "valid": (("sample",), np.array([False, True, True, False], dtype=bool)),
            },
            coords={"sample": np.arange(4, dtype=np.int64)},
            attrs={"fixture": "derivedRegistrarSource"},
        )
        matrix = pglDataMatrix(expectedMatrix.copy(deep=True), name="registrarMatrix")
        matrix.provenance = {
            "operation": "syntheticDerivedRegistrarFixture",
            "description": "Four synthetic values; no new scientific acquisition.",
            "sourceCheckpoint": dict(sourceReference),
        }
        session.dataMatrices = {"registrarMatrix": matrix}
        historyBeforeSave = historySnapshot(session)

        saveAttempted = True
        writeJSON("publicationAttempt.json", {
            "saveAttempted": True,
            "saveReturnedID": None,
            "sourceDatabaseID": databaseID,
            "rawParentSessionID": rawSessionID,
            "automaticRetryPermitted": False,
        })
        publicSQLWritesAllowed = True
        try:
            saveReturnedID = session.saveSession(
                settings=testSettings,
                checkpointName="derived registrar integration source",
                description="Synthetic child of raw registrar fixture; retained for export tests.",
            )
        finally:
            publicSQLWritesAllowed = False

        # Save this observation immediately: later verification may fail.
        writeJSON("publicationAttempt.json", {
            "saveAttempted": True,
            "saveReturnedID": saveReturnedID,
            "sourceDatabaseID": databaseID,
            "rawParentSessionID": rawSessionID,
            "automaticRetryPermitted": False,
            "note": "A missing return ID does not by itself prove no commit occurred.",
        })
        require(type(saveReturnedID) is int and saveReturnedID != rawSessionID,
                "Derived save did not return a new checkpoint ID.")
        require(historySnapshot(session) == historyBeforeSave, "Save rewrote action history.")
        recordPass("public source save returns a new committed derived checkpoint")

        reloaded = pglSession.loadSession(
            saveReturnedID, settings=testSettings, cacheDirectory=str(cacheRoot),
        )
        require(reloaded is not None, "Derived reload failed.")
        loadedSessions.append(reloaded)
        require(historySnapshot(reloaded) == historyBeforeSave, "Reload changed saved history.")
        require(
            reloaded.runs[0].data.uuid == acquisitionUUID
            and reloaded.runs[0].data.endTime == originalEndTime + 1.0,
            "Derived behavioral values failed round trip.",
        )
        require(set(reloaded.dataMatrices) == {"registrarMatrix"}, "Matrix attachment mismatch.")
        restoredMatrix = reloaded.dataMatrices["registrarMatrix"]
        restoredData = restoredMatrix.requireData()
        require(list(restoredData.xindexes) == [], "Loader created unexpected runtime indexes.")
        require(
            restoredData["signal"].chunks is not None
            and restoredData["valid"].chunks is not None,
            "Scientific variables were not lazy/chunked before computation.",
        )
        xr.testing.assert_identical(
            restoredData.compute(), expectedMatrix.drop_indexes("sample"),
        )
        require(restoredMatrix.provenance == matrix.provenance,
                "Matrix construction provenance changed.")
        recordPass("derived history, behavioral values, and lazy unindexed matrix round-trip")

        with postgres.connect() as connection:
            with connection.transaction():
                connection.execute(
                    "SET TRANSACTION ISOLATION LEVEL REPEATABLE READ, READ ONLY"
                )
                require(
                    pglDatabase._requireStableDatabaseIdentity(connection, postgres) == databaseID,
                    "Post-publication source identity mismatch.",
                )
                ancestry = pglDatabase._readExportAncestry(connection, postgres, saveReturnedID)
                require(
                    [item["sessionID"] for item in ancestry] == [rawSessionID, saveReturnedID],
                    "Derived source has unexpected ancestry.",
                )
                require(
                    ancestry[-1]["contractVersion"] == "pgl.derived.behavior.v3",
                    "Expected derived behavioral-v3 source fixture.",
                )
                files = pglDatabase._readExportFileInventory(
                    connection, postgres, ancestry[-1],
                )
                matrices = pglDatabase._readExportMatrixBundleInventory(
                    connection, postgres, ancestry[-1], requireAllChunks=True,
                )
                require(
                    len(matrices["revisions"]) == 1
                    and matrices["revisions"][0]["matrixRevisionID"] == restoredMatrix.matrixRevisionID,
                    "Unexpected registered matrix revision.",
                )
                require(
                    matrices["revisions"][0]["registeredRoot"] == str(payloadRoot),
                    "Matrix publication escaped the explicit fixture root.",
                )
                currentRaw = readSessionRow(connection, rawSessionID)
                require(currentRaw == rawCatalogBefore, "Original raw catalog row changed.")

        # Inspect the already restored history with the actual typed metadata.
        historyReport = pglDatabase._inspectExportHistoryReferences(
            reloaded,
            sourceDatabaseID=databaseID,
            sourceSessionIDs=[rawSessionID, saveReturnedID],
        )
        require(
            [item["checkpoint"]["sessionID"] for item in historyReport["checkpointReferences"]]
            == [rawSessionID],
            "Unexpected structured source-history references.",
        )
        require(
            historyReport["dependencyCompletenessEstablished"] is False,
            "History inspection overclaimed dependency completeness.",
        )
        recordPass("new source ancestry, contract, matrix bytes, and bounded history references verified")

        inspectedRaw = pglSession.loadSession(
            rawSessionID, settings=testSettings, cacheDirectory=str(cacheRoot),
        )
        require(inspectedRaw is not None, "Raw reread failed.")
        loadedSessions.append(inspectedRaw)
        require(
            inspectedRaw.runs[0].data.uuid == acquisitionUUID
            and inspectedRaw.runs[0].data.endTime == originalEndTime
            and inspectedRaw.actionHistory == [],
            "Original raw working contents changed.",
        )
        recordPass("original raw checkpoint still loads unchanged")

        revision = matrices["revisions"][0]
        writeJSON("matrixInventory.json", matrices)
        writeJSON("historyReferenceInspection.json", historyReport)
        # Native verifiedAt datetimes are deliberately excluded from this
        # portable fixture summary; catalog locations will be reread later.
        writeJSON("manifest.json", files["manifest"])
        writeJSON("derived-source-fixture.json", {
            "kind": "pglDisposableDerivedRegistrarSourceFixture",
            "version": 1,
            "environmentManifest": str(environmentPath),
            "rawFixtureManifest": str(rawFixturePath),
            "databaseName": databaseName,
            "databaseID": databaseID,
            "rawSessionID": rawSessionID,
            "derivedSessionID": saveReturnedID,
            "rawManifestHash": expectedRawHash,
            "derivedManifestHash": ancestry[-1]["manifestHash"],
            "contractVersion": ancestry[-1]["contractVersion"],
            "matrixName": "registrarMatrix",
            "matrixRevisionID": revision["matrixRevisionID"],
            "matrixBackendID": revision["backendID"],
            "matrixRegisteredRoot": revision["registeredRoot"],
            "matrixObjectKey": revision["objectKey"],
            "sourceProfileDisplayLabel": testSettings.name,
            "workspace": str(testPath),
            "payloadRoot": str(payloadRoot),
            "commitConfirmedByPublicSaveAndReadback": True,
            "destinationDatabaseAccessPerformed": False,
            "automaticDeletionPermitted": False,
        })

finally:
    # Close only readers owned by this test; never remove their backings.
    closeErrors = []
    for owner in reversed(loadedSessions):
        for matrix in owner.dataMatrices.values():
            try:
                matrix.close()
            except Exception as error:
                closeErrors.append(f"matrix: {type(error).__name__}: {error}")
        try:
            if owner.filesystem is not None:
                owner.filesystem.close()
        except Exception as error:
            closeErrors.append(f"filesystem: {type(error).__name__}: {error}")

    writeJSON("finalAttemptStatus.json", {
        "saveAttempted": saveAttempted,
        "saveReturnedID": saveReturnedID,
        "sourceCommitMayHaveOccurred": saveAttempted,
        "automaticRetryPermitted": False,
        "automaticDeletionPermitted": False,
        "readerCloseErrors": closeErrors,
    })
    rawAfter = snapshot(rawPayloadRoot)
    writeJSON("rawPayloadsAfter.json", rawAfter)
    require(rawAfter == rawBefore, "Original raw payload files/directories changed.")
    require(environmentPath.read_bytes() == environmentBytes, "Environment manifest changed.")
    require(rawFixturePath.read_bytes() == rawFixtureBytes, "Raw fixture manifest changed.")
    require(not closeErrors, "Reader close failures; see finalAttemptStatus.json.")
    recordPass("original raw payloads/manifests unchanged and owned readers closed")

print(f"\nAll {len(passed)} derived source-fixture checks passed.", flush=True)
'''

compile(testSource, str(scriptPath), "exec")
scriptPath.write_text(testSource, encoding="utf-8")
command = [
    sys.executable, str(scriptPath),
    str(repoPath), str(environmentPath), str(rawFixturePath), str(testPath),
]
(testPath / "command.json").write_text(json.dumps(command, indent=2))

print("AUTHORIZED: publish one derived synthetic SOURCE checkpoint beneath checkpoint 8.")
print("No destination database will be opened.")
print(f"Test source, payloads, and diagnostics retained at: {testPath}")
print("If a later check fails, source publication may already be committed. Do not retry blindly.")


def outputText(value):
    if isinstance(value, bytes): return value.decode("utf-8", errors="replace")
    return value or ""


try:
    completed = subprocess.run(
        command, cwd=repoPath, capture_output=True, text=True, timeout=600,
    )
except subprocess.TimeoutExpired as error:
    (testPath / "stdout.txt").write_text(outputText(error.stdout), encoding="utf-8")
    (testPath / "stderr.txt").write_text(outputText(error.stderr), encoding="utf-8")
    (testPath / "result.json").write_text(json.dumps({
        "status": "timeout",
        "sourceCommitMayHaveOccurred": True,
        "automaticRetryPermitted": False,
    }, indent=2))
    print(f"Source publication may have committed. Inspect {testPath} before retrying.")
    raise
except Exception as error:
    (testPath / "result.json").write_text(json.dumps({
        "status": "executionOrReportingError",
        "error": f"{type(error).__name__}: {error}",
        "automaticRetryPermitted": False,
    }, indent=2))
    raise
else:
    (testPath / "stdout.txt").write_text(completed.stdout, encoding="utf-8")
    (testPath / "stderr.txt").write_text(completed.stderr, encoding="utf-8")
    (testPath / "result.json").write_text(json.dumps({
        "status": "passed" if completed.returncode == 0 else "failed",
        "returnCode": completed.returncode,
        "automaticRetryPermitted": False,
        "note": "Failure does not imply source publication rolled back.",
    }, indent=2))
    print(completed.stdout)
    if completed.stderr:
        print("STDERR:")
        print(completed.stderr)
    print(f"All artifacts retained at: {testPath}")
    if completed.returncode:
        raise RuntimeError(
            f"Derived source-fixture test failed with exit code "
            f"{completed.returncode}. Inspect {testPath}; do not retry automatically."
        )

RuntimeError: Not executed. Set allowCommitDerivedSourceFixture = True to publish one derived synthetic source checkpoint beneath source checkpoint 8. No destination database will be opened; fixtures will be retained.